# LagSwitch Detector

This notebook processes Unreal Engine server logs and PCAP captures to detect lag-switch cheating behavior. Update the configuration values below before running the analysis cells.

## Setup

In [ ]:
# Optional: install dependencies if they are not already available.
# Uncomment and run the following line only when needed.
# %pip install matplotlib numpy protobuf pandas requests

### Imports & configuration

In [ ]:
from pathlib import Path
from datetime import datetime, timezone, timedelta
import sys
import os
import re
import json
import ast
import hashlib
import struct
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# --- Configuration ---
PROTO_PARSER_PATH = Path("libs")
PATH_LOGS = Path("data/")
CACHE_FILE = Path("cache/ip_cache_anonymized.json")
SERVER_IP = "137.74.44.5"

CACHE_FILE.parent.mkdir(parents=True, exist_ok=True)

if PROTO_PARSER_PATH.exists():
    sys.path.append(str(PROTO_PARSER_PATH))
    from ParserProtobufLogs.parser_utils import RingBufferParser
else:
    raise FileNotFoundError(f"PROTO_PARSER_PATH not found: {PROTO_PARSER_PATH}")

## Helper Functions

Generic pandas helpers used throughout the notebook (flattening protobuf-style dict/JSON columns, converting protobuf timestamps).

In [ ]:
def expand_string_dict_column(df: pd.DataFrame, column_name: str, suffix='_expanded') -> pd.DataFrame:
    """
    Expands a column containing dicts (or stringified dicts) into separate columns.
    Works whether the column contains actual dicts or stringified dicts.
    """
    df_copy = df.copy()

    # Convert strings to dicts if necessary
    def to_dict(x):
        if isinstance(x, str):
            try:
                import ast
                return ast.literal_eval(x)
            except (ValueError, SyntaxError):
                return {}
        elif isinstance(x, dict):
            return x
        else:
            return {}

    df_copy[column_name] = df_copy[column_name].apply(to_dict)

    # Expand the dict column
    expanded_df = pd.json_normalize(df_copy[column_name])

    # Avoid column collisions
    new_columns = [col + suffix if col in df_copy.columns else col for col in expanded_df.columns]
    expanded_df.columns = new_columns

    # Join back
    df_copy = df_copy.drop(columns=[column_name]).join(expanded_df)

    return df_copy


def convert_ts(ts):
    # Return NaT for invalid rows
    if not isinstance(ts, dict):
        return pd.NaT

    seconds = ts.get("seconds")
    nanos = ts.get("nanos", 0)

    # Validate seconds
    if seconds is None:
        return pd.NaT

    try:
        # Build datetime without floating point arithmetic

        return (
            datetime.fromtimestamp(seconds, tz=timezone.utc)
            + timedelta(microseconds=nanos / 1000)
        )
    except Exception:
        return pd.NaT


def compute_ts(df):
    df = df.dropna(subset=["ts"])
    df["timestamp"] = df["ts"].apply(convert_ts)
    return_df = df.drop(columns=["ts"])
    return return_df


def expand_payload_column(df: pd.DataFrame, column_name: str, drop_original=True, suffix='_expanded'):
    """
    Safely expands a column containing dicts or stringified dicts/JSON into separate columns.
    Handles nested dictionaries and ensures alignment with original rows.
    """
    df_copy = df.copy()

    def to_dict(x):
        if isinstance(x, dict):
            return x
        elif isinstance(x, str):
            try:
                return ast.literal_eval(x)  # Python-style dict
            except (ValueError, SyntaxError):
                try:
                    return json.loads(x)     # JSON-style
                except (ValueError, json.JSONDecodeError):
                    return {}
        else:
            return {}

    # Convert all entries to dict
    df_copy[column_name] = df_copy[column_name].apply(to_dict)

    # Only expand rows that are actual dicts
    mask = df_copy[column_name].apply(lambda x: isinstance(x, dict) and bool(x))
    if mask.any():
        expanded = pd.json_normalize(df_copy.loc[mask, column_name])
        # Keep original indices to align properly
        expanded.index = df_copy.loc[mask].index

        # Rename columns if collision
        new_cols = [col + suffix if col in df_copy.columns else col for col in expanded.columns]
        expanded.columns = new_cols

        # Join back
        df_copy = df_copy.join(expanded)

    # Drop original column if requested
    if drop_original:
        df_copy = df_copy.drop(columns=[column_name])

    return df_copy

## 1. Parse Server Logs

### 1.1 Extract raw log entries from traces

Function to extract the logs:

In [ ]:
def extract_logs(folder_path):
    # Regex to extract timestamp
    pattern = r"trace_server_(\d{4}\.\d{2}\.\d{2}-\d{2}\.\d{2}\.\d{2})(?:_anonymized)?\.bin"

    dataframes_dict = {}

    for filename in os.listdir(folder_path):
        match = re.match(pattern, filename)
        
        if match:
            timestamp_str = match.group(1)
            
            # Convert to datetime (optional but recommended)
            timestamp = datetime.strptime(timestamp_str, "%Y.%m.%d-%H.%M.%S")
            
            file_path = os.path.join(folder_path, filename)
            
            # Parse file
            parser_server_logs = RingBufferParser.from_file(file_path)
            server_logs_df = parser_server_logs.to_dataframe()
            server_logs_df = server_logs_df[server_logs_df["ts"].notna()]  # Drop rows with invalid timestamps
            
            # Store in dict
            dataframes_dict[timestamp] = server_logs_df

    return dataframes_dict

### 1.2 Extract players

In [ ]:
def compute_base_name(df_players):
    df_players["base_name"] = df_players["name_expanded"].str.rsplit("-", n=1).str[0]
    return df_players

def extract_players(server_logs):
    players_dict = {}

    for ts, df in server_logs.items():
        df = df[df["event_type"] == "new_player"]
        df = expand_string_dict_column(df, "payload")
        df = expand_payload_column(df, "new_player")
        df = df[df["event_type"] == "new_player"]
        df = df[["timestamp", "player_id_expanded", "name_expanded", "ip_port", "update_ip_port"]]
        df = compute_base_name(df)
        players_dict[ts] = df

    return players_dict


def extract_players_multiple_sessions(sessions_folder):
    print("Starting recursive folder processing...")

    merged_pcap_dict = {}
    burst_sum = pd.DataFrame()  # Initialize burst summary DF

    # --------------------------------------------------
    # 1. Traverse folders
    # --------------------------------------------------
    for root, dirs, files in os.walk(sessions_folder):
        if "Server" in dirs:
            server_folder = os.path.join(root, "Server")
            print(f"Processing: {server_folder}")

            # --------------------------------------------------
            # 2. Build enriched PCAP dict (your pipeline)
            # --------------------------------------------------
            logs = extract_logs(server_folder)
            pcap_dict = extract_players(logs)

            if pcap_dict is None:
                continue

            # --------------------------------------------------
            # 3. Merge into global dict
            # --------------------------------------------------
            for ts, df in pcap_dict.items():

                if df is None or df.empty:
                    continue

                if ts not in merged_pcap_dict:
                    merged_pcap_dict[ts] = df
                else:
                    merged_pcap_dict[ts] = pd.concat(
                        [merged_pcap_dict[ts], df],
                        ignore_index=True
                    )

    print("Finished merging all sessions.")

    # --------------------------------------------------
    # 4. Plot CDF by region
    # --------------------------------------------------
    # plot_iat_cdf_by_region(merged_pcap_dict, tail_percentile=tail_percentile)    
    

    return merged_pcap_dict, burst_sum

### 1.3 Extract playing phases
Phases are Warmup, Playing and PostGame.

In [ ]:
def extract_phases(server_logs):
    phase_dict = {}

    for ts, df in server_logs.items():
        df = df[df["event_type"] == "phase_event"]
        df = expand_string_dict_column(df, "payload")
        df = expand_payload_column(df, "phase_event")
        df = df[df["event_type"] == "phase_event"]
        df = df[["timestamp", "phase_name_expanded", "start"]].sort_values("timestamp")
        # Split starts and stops
        starts = df[df["start"]].copy()
        stops = df[~df["start"]].copy()

        # Prepare for pairing
        starts = starts.rename(columns={"timestamp": "start_time"})
        stops = stops.rename(columns={"timestamp": "stop_time"})

        starts["idx"] = starts.groupby("phase_name_expanded").cumcount()
        stops["idx"] = stops.groupby("phase_name_expanded").cumcount()

        # Merge directly
        merged = starts.merge(
            stops,
            on=["phase_name_expanded", "idx"],
            how="left"
        )

        phase_dict[ts] = merged[["phase_name_expanded", "start_time", "stop_time"]]
        
    return phase_dict

### 1.4 Extract cheat intervals

In [ ]:
def extract_cheat_intervals(df):
    # Filter cheat events
    df = df[df["entry_type"] == "event"]
    df = expand_payload_column(df, "payload")
    df = df[df["event_type"] == "cheat"]
 
    # Sort by timestamp
    df = df.sort_values("timestamp")  # adjust if your timestamp column has a different name
 
    result = {}
    last_ts = df["timestamp"].max()  # last event timestamp
    for _, row in df.iterrows():
        player = row["cheater_id"]
        if row["type"] == "DOS":
            player = row["target_id"]  # For DOS, we track the victim instead of the cheater
        cheat_type = row["type"]
        is_start = row["start"]
        ts = row["timestamp"]
        result.setdefault(player, {})
        result[player].setdefault(cheat_type, [])
        if is_start:
 
            # Start a new interval
            result[player][cheat_type].append({"start": ts, "end": None, "duration": None})
            if cheat_type == "DOS":
                # @todo : Check if we need this 
                duration = row["dur"] if pd.notna(row.get("dur")) else 10
                end_ts = ts + pd.Timedelta(seconds=duration)
                result[player][cheat_type][-1]["end"] = end_ts
                result[player][cheat_type][-1]["duration"] = duration
        else:
            # Close the last open interval
            intervals = result[player][cheat_type]
 
            for interval in reversed(intervals):
                if interval["end"] is None:
                    interval["end"] = ts
                    interval["duration"] = (interval["end"] - interval["start"]).total_seconds()
                    break
    # Close any remaining open intervals with last timestamp
    # (special case: LagSwitch intervals that are still open after processing
    # every event are capped at start + 5 seconds instead of stretching all
    # the way to last_ts, since a lag switch is assumed to never run that long
    # if its closing event was missing/dropped)
    for player in result:
        for cheat_type in result[player]:
            for interval in result[player][cheat_type]:
                if interval["end"] is None:
                    if cheat_type == "LagSwitch":
                        interval["end"] = interval["start"] + pd.Timedelta(seconds=5)
                    else:
                        interval["end"] = last_ts
                    interval["duration"] = (interval["end"] - interval["start"]).total_seconds()
    return result
 
def extract_cheats(server_logs):
    cheats_dict = {
        ts: extract_cheat_intervals(df)
        for ts, df in server_logs.items()
    }
    return cheats_dict

### 1.5 Extract network traffic traces from PCAPs

In [ ]:
import re
from pathlib import Path
from datetime import datetime
import pandas as pd
import subprocess
from multiprocessing import Pool, cpu_count

class BitReader:
    def __init__(self, data: bytes):
        self.data = data
        self.num_bits = len(data) * 8
        self.bit_pos = 0

    def read_bits(self, n: int) -> int:
        if self.bit_pos + n > self.num_bits:
            raise ValueError("Read past end")

        value = 0
        bits_read = 0

        while bits_read < n:
            byte_index = self.bit_pos // 8
            bit_offset = self.bit_pos % 8

            remaining_in_byte = 8 - bit_offset
            bits_to_read = min(n - bits_read, remaining_in_byte)

            current_byte = self.data[byte_index]

            mask = (1 << bits_to_read) - 1
            extracted = (current_byte >> bit_offset) & mask

            value |= extracted << bits_read

            self.bit_pos += bits_to_read
            bits_read += bits_to_read

        return value

    def read_bit(self) -> int:
        return self.read_bits(1)

    def read_bool(self) -> bool:
        return bool(self.read_bits(1))

    def align_to_byte(self):
        self.bit_pos = (self.bit_pos + 7) & ~7

    def peek_bits(self, n: int) -> int:
        old_pos = self.bit_pos
        value = self.read_bits(n)
        self.bit_pos = old_pos
        return value

    def remaining_bits(self):
        return self.num_bits - self.bit_pos

    def at_end(self):
        return self.bit_pos >= self.num_bits
    

def hash_bitstream(data: bytes, num_bits: int) -> bytes:
    hasher = hashlib.sha1()

    num_bytes = num_bits // 8
    remaining_bits = num_bits % 8

    # 1. Full bytes
    if num_bytes > 0:
        hasher.update(data[:num_bytes])

    # 2. Remaining bits (masked)
    if remaining_bits > 0:
        last_byte = data[num_bytes]

        mask = (1 << remaining_bits) - 1
        last_byte &= mask

        hasher.update(bytes([last_byte]))

    # 3. Hash NumBits as int32 (IMPORTANT: little-endian like Unreal)
    hasher.update(struct.pack('<i', num_bits))

    return hasher.digest()

def compute_flow_id(src_ip, src_port, dst_ip, dst_port):
    """Create a bidirectional flow identifier."""
    a = (src_ip, src_port)
    b = (dst_ip, dst_port)
    if a <= b:
        return f"{src_ip}:{src_port}-{dst_ip}:{dst_port}"
    else:
        return f"{dst_ip}:{dst_port}-{src_ip}:{src_port}"
    
def compute_direction(src_ip, dst_ip):
    """Determine the direction of a flow based on IP addresses."""
    if src_ip == dst_ip:
        return "loopback"
    elif src_ip == "137.74.44.5":
        return "outgoing"
    else:
        return "incoming"

# @warning: Not filtered by port actually, port 7777 (server_port) and 7778 enebaled
def process_all_pcaps(path, server_port=7777):

    folder = Path(path)

    # Collect PCAP files
    pattern = r"^capture_server_(\d{4}\.\d{2}\.\d{2}-\d{2}\.\d{2}\.\d{2})(?:_anonymized)?\.pcap$"

    tasks = []

    for file in folder.glob("capture_server_*.pcap"):
        match = re.match(pattern, file.name)
        if match:
            ts = datetime.strptime(match.group(1), "%Y.%m.%d-%H.%M.%S")
            tasks.append((ts, file))

    print(f"Found {len(tasks)} PCAP files to process.")

    def get_conn_info(payload):
        if not payload:
            return None

        reader = BitReader(payload)

        session_id = reader.read_bits(2)
        client_id = reader.read_bits(3)

        # Keep alignment consistent with original (6 bits total)
        handshake = bool(reader.read_bits(1))

        return {
            "session_id": session_id,
            "client_id": client_id,
            "handshake": handshake,
            "restart_handshake": None  # still unknown / not parsed here
        }, reader.bit_pos  # return current bit position for next parsing

    # Header extraction function
    def get_header_info(payload):
        if len(payload) < 5:
            return None, None

        header = int.from_bytes(payload[0:5], byteorder="little")
        header = (header >> 6) & 0xffffffff

        Seq = (header >> 18) & 0x3fff
        AckedSeq = (header >> 4) & 0x3fff

        HistoryWordCountMask = (1 << 4) - 1
        HistoryWordCount = (header & HistoryWordCountMask) + 1

        HistoryWords = []
        lenWords = 4

        for i in range(HistoryWordCount):
            start = 4 + lenWords * i
            stop = start + lenWords
            if stop > len(payload):
                break

            word = int.from_bytes(payload[start:stop], byteorder="little")
            HistoryWords.append(word)

        pos = 38 + (HistoryWordCount - 1) * lenWords * 8

        info = {
            "Seq": Seq,
            "AckedSeq": AckedSeq,
            "HistoryWordCount": HistoryWordCount,
            "HistoryWords": HistoryWords,
        }

        return info, pos

    # Function to parse one PCAP fully
    def process_pcap_file_tshark(file):
        cmd = [
            "tshark", "-r", str(file),
            "-Y", f"udp.port == {server_port} || udp.port == 7778 || udp.port == 7788",
            "-T", "fields",
            "-e", "frame.time_epoch",
            "-e", "ip.src",
            "-e", "ip.dst",
            "-e", "udp.srcport",
            "-e", "udp.dstport",
            "-e", "data"
        ]

        process = subprocess.Popen(cmd, stdout=subprocess.PIPE, text=True)

        rows = []

        for line in process.stdout:   # stream line-by-line
            line = line.strip()
            if not line:
                continue

            parts = line.split("\t")
            if len(parts) < 6:
                continue

            timestamp, src_ip, dst_ip, src_port, dst_port, payload_hex = parts

            if not payload_hex:
                continue

            try:
                payload_bytes = bytes.fromhex(payload_hex)
            except ValueError:
                continue

            try:
                conn_info, _ = get_conn_info(payload_bytes)
                if conn_info is None:
                    continue

                header_info, _ = get_header_info(payload_bytes)
                if header_info is None:
                    continue

            except ValueError:
                print(f"Error parsing packet at {timestamp} from {src_ip}:{src_port} to {dst_ip}:{dst_port}")
                continue

            except ValueError:
                print(f"Error parsing packet at {timestamp} from {src_ip}:{src_port} to {dst_ip}:{dst_port}")
                # BitReader overflow → packet too small or unexpected format
                continue

            rows.append({
                "timestamp": float(timestamp),
                "src_ip": src_ip,
                "dst_ip": dst_ip,
                "src_port": int(src_port),
                "dst_port": int(dst_port),
                "payload": payload_hex,
                "client_id": conn_info["client_id"],
                "session_id": conn_info["session_id"],
                "handshake": conn_info["handshake"],
                "restart_handshake": conn_info["restart_handshake"],
                "Seq": header_info["Seq"],
                "AckedSeq": header_info["AckedSeq"],
                "HistoryWordCount": header_info["HistoryWordCount"],
                "HistoryWords": header_info["HistoryWords"]
            })

        process.stdout.close()
        process.wait()

        df = pd.DataFrame(rows)
        if not df.empty:
            df["timestamp"] = pd.to_datetime(df["timestamp"], unit="s", utc=True)

        return df
    def process_pcap_file_with_flows(file, server_ip):
        df = process_pcap_file_tshark(file)  # your function that extracts payload, Seq, etc.
        if df.empty:
            return df

        # Add flow ID
        df["flow_id"] = df.apply(lambda r: compute_flow_id(r["src_ip"], r["src_port"], r["dst_ip"], r["dst_port"]), axis=1)

        # Add direction
        df["direction"] = df.apply(lambda r: compute_direction(r["src_ip"], r["dst_ip"]), axis=1)


        return df

    # Wrapper for multiprocessing to show progress
    def process_one(args):
        ts, file = args
        df = process_pcap_file_with_flows(file, server_ip="137.74.44.5")
        return ts, df, f"{file.name}: {len(df)} packets"


    # Run multiprocessing pool
    pcap_df_dict = {}

    # tasks = tasks[:4]
    total_files = len(tasks)

    # with Pool(min(4, cpu_count())) as pool:  # limit workers to avoid RAM spikes
    #     for i, (ts, df, msg) in enumerate(pool.imap_unordered(process_one, tasks), 1):
    #         pcap_df_dict[ts] = df
    #         print(f"[{i}/{total_files}] {msg}")

    for i, task in enumerate(tasks, 1):
        ts, df, msg = process_one(task)
        pcap_df_dict[ts] = df
        print(f"[{i}/{total_files}] {msg}")

    print("All PCAPs processed!")

    return pcap_df_dict

Compute IATs (inter-arrival times)

In [ ]:
def add_iat_to_pcaps(pcap_df_dict):
    print("Adding IATs (strict + raw)...")

    for ts, df in pcap_df_dict.items():
        if df.empty:
            continue

        # Ensure timestamp is datetime (no sorting!)
        if not pd.api.types.is_datetime64_any_dtype(df["timestamp"]):
            df["timestamp"] = pd.to_datetime(df["timestamp"])

        # --------------------------------------------------
        # 1. Sequence-based validity
        # --------------------------------------------------
        df["prev_seq"] = df.groupby(["flow_id", "direction"])["Seq"].shift(1)
        df["seq_ok"] = (df["Seq"] == df["prev_seq"] + 1)

        # --------------------------------------------------
        # 2. RAW IAT (no filtering, no sequence constraint)
        # --------------------------------------------------
        df["iat_raw"] = (
            df.groupby(["flow_id", "direction"])["timestamp"]
            .diff()
            .dt.total_seconds()
        )

        # --------------------------------------------------
        # 3. STRICT IAT (sequence-valid only)
        # --------------------------------------------------
        df["iat"] = df["iat_raw"].copy()
        df.loc[~df["seq_ok"], "iat"] = None

        # Optional cleanup
        df.drop(columns=["prev_seq"], inplace=True)

        print(
            f"{ts}: "
            f"strict={df['iat'].notna().sum()} / {len(df)}, "
            f"raw={df['iat_raw'].notna().sum()}"
        )

    return pcap_df_dict

### 1.6 Add decayed (adaptive) IAT parameters

In [ ]:
import numpy as np


def decay_iat_features(df, half_life=1.0, eps=1e-8):
    """
    Symmetric version (no log scaling).
    Adds:
        - dk_iat_mean
        - dk_iat_std
        - dk_iat_mean_filt   (updated only if xi within ±2 std)
        - dk_iat_std_filt
        - dk_iat_accepted    ("init", "warmup", "accepted", "rejected", or None if xi is nan)
    """
    df["dk_iat_mean"]      = np.nan
    df["dk_iat_std"]       = np.nan
    df["dk_iat_mean_filt"] = np.nan
    df["dk_iat_std_filt"]  = np.nan
    df["dk_iat_accepted"]  = None

    lambd = np.log(2) / half_life
    df_in = df[df["direction"] == "incoming"].copy()
    df_in = df_in.sort_values(["flow_id", "timestamp"])

    for flow_id, g in df_in.groupby("flow_id"):
        g = g.sort_values("timestamp")
        t = g["timestamp"].astype("int64") / 1e9
        x = g["iat_raw"].values

        S = Q = W = 0.0
        Sf = Qf = Wf = 0.0
        mean_f = std_f = np.nan
        n_accepted = 0

        mean_list      = []
        std_list       = []
        mean_filt_list = []
        std_filt_list  = []
        accepted_list  = []

        prev_t = None

        for i in range(len(g)):
            dt     = 0.0 if prev_t is None else t.iloc[i] - prev_t
            prev_t = t.iloc[i]
            decay  = np.exp(-lambd * dt)

            S  *= decay;  Q  *= decay;  W  *= decay
            Sf *= decay;  Qf *= decay;  Wf *= decay

            xi = x[i]

            # --- main update ---
            if np.isfinite(xi):
                S += xi
                Q += xi * xi
                W += 1

            if W > 0:
                mean = S / W
                std  = np.sqrt(max((Q / W) - (S / W) ** 2, 0.0))
            else:
                mean = std = np.nan

            # --- filtered update (±2 std gate) ---
            label = None
            if np.isfinite(xi):
                if n_accepted == 0:
                    Sf = xi;  Qf = xi * xi;  Wf = 1.0
                    n_accepted = 1
                    mean_f = Sf / Wf
                    std_f  = np.sqrt(max((Qf / Wf) - (Sf / Wf) ** 2, 0.0))
                    label  = "init"

                elif n_accepted < 190:
                    Sf += xi;  Qf += xi * xi;  Wf += 1
                    n_accepted += 1
                    mean_f = Sf / Wf
                    std_f  = np.sqrt(max((Qf / Wf) - (Sf / Wf) ** 2, 0.0))
                    label  = "warmup"

                else:
                    effective_std = max(std_f, 0.001)
                    if abs(xi - mean_f) <= 2.0 * effective_std:
                        Sf += xi;  Qf += xi * xi;  Wf += 1
                        n_accepted += 1
                        mean_f = Sf / Wf
                        std_f  = np.sqrt(max((Qf / Wf) - (Sf / Wf) ** 2, 0.0))
                        label  = "accepted"
                    else:
                        label  = "rejected"

            mean_list.append(mean)
            std_list.append(std)
            mean_filt_list.append(mean_f)
            std_filt_list.append(std_f)
            accepted_list.append(label)

        df.loc[g.index, "dk_iat_mean"]      = mean_list
        df.loc[g.index, "dk_iat_std"]       = std_list
        df.loc[g.index, "dk_iat_mean_filt"] = mean_filt_list
        df.loc[g.index, "dk_iat_std_filt"]  = std_filt_list
        df.loc[g.index, "dk_iat_accepted"]  = accepted_list

    return df

def dk_on_dict(session_dict, half_life=1.0):
    return {
        session_ts: decay_iat_features(df, half_life)
        for session_ts, df in session_dict.items()
    }

### 1.7 Label packets
Attach phase, cheat, and geo-IP labels to every packet.

In [ ]:
def label_phases_in_packets(packet_dict, phase_dict):
    print("Labeling phases in dataframes...")

    for ts, df in packet_dict.items():

        phases_df = phase_dict.get(ts)
        if phases_df is None or df.empty:
            print(f"No phase data for timestamp {ts}")
            continue

        # Ensure time column is datetime
        if not pd.api.types.is_datetime64_any_dtype(df["timestamp"]):
            df["timestamp"] = pd.to_datetime(df["timestamp"])

        # Initialize column
        df["phase"] = None

        for _, row in phases_df.iterrows():
            start = row["start_time"]
            stop = row.get("stop_time")

            mask = df["timestamp"] >= start
            if pd.notna(stop):
                mask &= df["timestamp"] <= stop

            # Map phase name
            phase_name = row["phase_name_expanded"]

            if phase_name.endswith("Warmup"):
                df.loc[mask, "phase"] = "warmup"
            elif phase_name.endswith("Playing"):
                df.loc[mask, "phase"] = "playing"
            elif phase_name.endswith("PostGame"):
                df.loc[mask, "phase"] = "postgame"

        print(f"Timestamp {ts}: labeled {df['phase'].notna().sum()} rows")

    return packet_dict


def label_cheats_in_packets(pcap_dict, players_dict, cheats_dict, server_ip="137.74.44.5", inlab=False):
    print("Labeling cheating activity in packets...")

    for ts, df in pcap_dict.items():
        if df.empty:
            continue
        
        # Initialize labels
        df["is_cheating"] = False
        df["cheat_type"] = None

        

        players_df = players_dict.get(ts)
        cheats_ts = cheats_dict.get(ts, {})

        if players_df is None or not cheats_ts:
            print(f"Skipping {ts}: missing players or cheats data")
            continue

        game_end = pd.to_datetime(df["timestamp"].max(), utc=True).tz_convert(None)
        # =========================================================
        # 1. Ensure timestamps are datetime
        # =========================================================
        if not pd.api.types.is_datetime64_any_dtype(df["timestamp"]):
            df["timestamp"] = pd.to_datetime(df["timestamp"]).astype("datetime64[ns, UTC]")

        if not pd.api.types.is_datetime64_any_dtype(players_df["timestamp"]):
            players_df["timestamp"] = pd.to_datetime(players_df["timestamp"]).astype("datetime64[ns, UTC]")

        # =========================================================
        # 2. Build IP:port mapping table
        # =========================================================
        
        # Normalize timestamps BEFORE building ip_port_intervals
        df["timestamp"] = pd.to_datetime(df["timestamp"], utc=True).dt.tz_convert(None).astype("datetime64[ns]")
        players_df["timestamp"] = pd.to_datetime(players_df["timestamp"], utc=True).dt.tz_convert(None).astype("datetime64[ns]")

        
        ip_port_intervals = (
            players_df[["timestamp", "ip_port", "player_id_expanded"]]
            .dropna(subset=["ip_port"])
            .sort_values("timestamp")
        )

        # =========================================================
        # 3. Vectorized client IP extraction
        # =========================================================
        src = df["src_ip"] + ":" + df["src_port"].astype(str)
        dst = df["dst_ip"] + ":" + df["dst_port"].astype(str)

        df["ip_port"] = np.where(src != server_ip, src, dst)

        # =========================================================
        # 4. Sort before merge_asof
        # =========================================================
        df = df.sort_values("timestamp")

        # =========================================================
        # 5. Time-aware join (FAST replacement for row-wise lookup)
        # =========================================================

        df = pd.merge_asof(
            df,
            ip_port_intervals,
            by="ip_port",
            left_on="timestamp",
            right_on="timestamp",
            direction="backward"
        )

        # Assign player_id
        df["player_id"] = df["player_id_expanded"]

        # =========================================================
        # 6. (Optional) Clean up
        # =========================================================
        df.drop(columns=["player_id_expanded"], inplace=True)

        

        # =========================================================
        # 4. Apply cheat intervals per player
        # =========================================================
        for player_id, cheat_types in cheats_ts.items():

            player_mask = df["player_id"] == player_id
            if not player_mask.any():
                continue

            for cheat_name, intervals in cheat_types.items():
                for interval in intervals:
                    if cheat_name in ("LagSwitch", "FixedDelay", "DOS"):

                        start = interval["start"]
                        end = interval["end"]

                        # =========================================================
                        # Normalize FIRST before any comparisons
                        # =========================================================
                        if hasattr(start, "tzinfo") and start.tzinfo is not None:
                            start = start.tz_convert(None)
                        if hasattr(end, "tzinfo") and end.tzinfo is not None:
                            end = end.tz_convert(None)

                        # =========================================================
                        # Check if cheat was cut short by game end
                        # =========================================================
                        grace_period = pd.Timedelta(seconds=5 if cheat_name in ("LagSwitch", "FixedDelay") else 10)

                        

                        if interval["duration"] == 0: 
                            end = start + grace_period
                            if end > game_end:
                                if start >= game_end - grace_period:
                                    print(
                                        f"WARNING: {cheat_name} for player {player_id} at {ts} "
                                        f"started only {(game_end - start).total_seconds():.2f}s before game end — skipping"
                                    )
                                    continue
                                else:
                                    print(
                                        f"INFO: {cheat_name} for player {player_id} at {ts} "
                                        f"was cut short by game end (clamped {end} → {game_end})"
                                    )
                                    end = game_end
                        
                        # only activate this on in lab recordings
                        if inlab == True and cheat_name == "LagSwitch" and end - start < pd.Timedelta(seconds=2):
                            start -= pd.Timedelta(seconds=5)

                        time_mask = (
                            (df["timestamp"] >= start) &
                            (df["timestamp"] <= end)
                        )

                        mask = player_mask & time_mask

                        # After applying the mask, check if it matched any packets
                        if not mask.any():
                            print(f"WARNING: {cheat_name} for player {player_id} at {ts} "
                                f"[{start} → {end}] matched 0 packets")
                            continue  # optional, skip labeling since there's nothing to label

                        # label cheating
                        df.loc[mask, "is_cheating"] = True

                        df.loc[mask, "cheat_type"] = df.loc[mask, "cheat_type"].apply(
                            lambda x: cheat_name if x is None else (x if cheat_name in x.split("|") else f"{x}|{cheat_name}")
                        )
                        
        pcap_dict[ts] = df  
        print(f"{ts}: labeled {df['is_cheating'].sum()} cheating packets")

    return pcap_dict

Enrich packets with geo-IP / ISP info (results are cached to `CACHE_FILE`; the shipped cache is anonymized).

In [ ]:
import requests
import json
import os
import time

# --- 1. Extract client IP ---
def extract_client_ip(flow_id):
    try:
        left, right = flow_id.split("-")

        ip1 = left.split(":")[0]
        ip2 = right.split(":")[0]

        return ip2 if ip1 == SERVER_IP else ip1

    except Exception:
        return None


# --- 2. Load cache ---
def load_cache():
    if os.path.exists(CACHE_FILE):
        with open(CACHE_FILE, "r") as f:
            return json.load(f)
    return {}


# --- 3. Save cache ---
def save_cache(cache):
    with open(CACHE_FILE, "w") as f:
        json.dump(cache, f, indent=2)


# --- 4. GeoIP lookup ---
def ip_lookup(ip):
    try:
        r = requests.get(f"http://ip-api.com/json/{ip}", timeout=2)
        data = r.json()

        if data["status"] == "success":
            # to distinguish between Benin and Cameroun Litoral
            regionName =  data.get("regionName")
            if regionName == "Littoral":
                regionName = data.get("country") + '_' + regionName

            return {
                "country": data.get("country"),
                "region": data.get("regionName"),
                "isp": data.get("isp"),
            }
        else:
            return None

    except Exception:
        return None


# --- 5. Main function ---
def enrich_with_geo(df):

    df = df.copy()

    # extract IP
    df["client_ip"] = df["flow_id"].apply(extract_client_ip)

    # load cache
    cache = load_cache()

    unique_ips = df["client_ip"].dropna().unique()
    print(f"{len(unique_ips)} unique IPs")

    new_lookups = 0

    for ip in unique_ips:
        if ip not in cache:
            cache[ip] = ip_lookup(ip)
            new_lookups += 1

            # avoid rate limiting (~45 req/min)
            time.sleep(0.5)

    print(f"{new_lookups} new API calls")

    # save updated cache
    save_cache(cache)

    # --- map back ---
    def get_field(ip, field):
        if ip in cache and cache[ip]:
            return cache[ip].get(field)
        return None

    df["country"] = df["client_ip"].apply(lambda ip: get_field(ip, "country"))
    df["region"] = df["client_ip"].apply(lambda ip: get_field(ip, "region"))
    df["isp"] = df["client_ip"].apply(lambda ip: get_field(ip, "isp"))

    return df

def enrich_all_geo(pcap_dict):
    for ts, df in pcap_dict.items():
        if df is None or df.empty:
            print("df None in enrich with geo")
            continue

        pcap_dict[ts] = enrich_with_geo(df)

    return pcap_dict

In [ ]:
def get_pcap_dict_labelled(folderpath, half_life=1.0, inlab=False):
    # Extract logs
    server_logs = extract_logs(folderpath)

    players_dict = extract_players(server_logs)
    
    phases_dict = extract_phases(server_logs)

    cheats_dict = extract_cheats(server_logs)

    # extract packets from PCAP
    pcap_dict = process_all_pcaps(folderpath, server_port=7777)

    # compute inter-arrival times (IATs) for each packet
    pcap_dict = add_iat_to_pcaps(pcap_dict)

    pcap_dict = dk_on_dict(pcap_dict, half_life)

    pcap_dict = label_phases_in_packets(pcap_dict, phases_dict)

    pcap_dict = label_cheats_in_packets(pcap_dict, players_dict, cheats_dict, inlab=inlab)
    
    pcap_dict = enrich_all_geo(pcap_dict)

    return pcap_dict

In [ ]:
def gather_all_pcaps_with_iats(sessions_folder, tail_percentile=None, half_life=1.0, inlab=False):
    print("Starting recursive PCAP processing...")

    merged_pcap_dict = {}
    burst_sum = pd.DataFrame()  # Initialize burst summary DF

    # --------------------------------------------------
    # 1. Traverse folders
    # --------------------------------------------------
    for root, dirs, files in os.walk(sessions_folder):
        if "Server" in dirs:
            server_folder = os.path.join(root, "Server")
            print(f"Processing: {server_folder}")

            # --------------------------------------------------
            # 2. Build enriched PCAP dict (your pipeline)
            # --------------------------------------------------
            pcap_dict = get_pcap_dict_labelled(server_folder, half_life, inlab=inlab)

            if pcap_dict is None:
                continue

            # --------------------------------------------------
            # 3. Merge into global dict
            # --------------------------------------------------
            for ts, df in pcap_dict.items():

                if df is None or df.empty:
                    continue

                if ts not in merged_pcap_dict:
                    merged_pcap_dict[ts] = df
                else:
                    merged_pcap_dict[ts] = pd.concat(
                        [merged_pcap_dict[ts], df],
                        ignore_index=True
                    )

    print("Finished merging all sessions.")

    # --------------------------------------------------
    # 4. Plot CDF by region
    # --------------------------------------------------
    # plot_iat_cdf_by_region(merged_pcap_dict, tail_percentile=tail_percentile)    
    

    return merged_pcap_dict

Run the full parsing pipeline over every session in `PATH_LOGS`:

In [ ]:
pcap_inthewild = gather_all_pcaps_with_iats(PATH_LOGS, half_life=10)

## 2. Detection

### 2.1 Event-level detection (adaptive detector)

The core detector: for each flow, whenever an IAT gap exceeds an adaptive threshold (derived from that flow's decayed mean/stdev), check whether the following packets arrive suspiciously fast (a "burst" that fills in the missing time) and flag the window if so.

In [ ]:
def extract_suspicious_window_packet_per_packet(
    packet_dict,
    gap_threshold_min_ms=400,
    burst_threshold_max_ms=1,
    remove_fd=False,
    remove_dos=False,
    suspicious_fill_ratio_threshold=0.5,
    max_dk_mean_ms=None,
):
 
    all_df = []
    # Keep track of which (dict_key, original_index) each concatenated row came from,
    # so we can write the detector verdict back into the *original* packet_dict dataframes.
    source_keys = []
    source_orig_idx = []
 
    for key, df in packet_dict.items():
        if df is not None and not df.empty:
            all_df.append(df)
            source_keys.extend([key] * len(df))
            source_orig_idx.extend(df.index.tolist())
 
    if not all_df:
        print("No data available")
        return packet_dict
 
    df = pd.concat(all_df, ignore_index=True)
    df["_src_key"] = source_keys
    df["_src_idx"] = source_orig_idx
 
    mask = (
        (df["phase"] == "playing") &
        (df["iat"].notna()) &
        (df["region"].notna()) &
        (df["dk_iat_mean_filt"].notna())
    )
 
    if remove_fd:
        mask &= (df["is_cheating"] == False) | (df["cheat_type"] != "FixedDelay")
 
    if remove_dos:
        mask &= (df["is_cheating"] == False) | (df["cheat_type"] != "DOS")
 
    # Exclude packets whose baseline connection is already too unstable to
    # meaningfully judge for lag-switching (dk_iat_mean_filt is in seconds,
    # so convert the ms threshold to match before comparing).
    if max_dk_mean_ms is not None:
        mask &= df["dk_iat_mean"] <= (max_dk_mean_ms / 1000)
 
    df = df[mask].copy()
 
    df["iat_ms"]      = df["iat"] * 1000
    df["dk_mean_ms"]  = df["dk_iat_mean_filt"] * 1000
    df["dk_stdev_ms"] = df["dk_iat_std_filt"] * 1000
 
    def get_lower_threshold(mean, stdev):
        # mean, stdev in ms
        threshold = mean - 2 * stdev
        return max(threshold, burst_threshold_max_ms)
 
    def get_upper_threshold(mean, stdev):
        # mean, stdev in ms
        threshold = mean + 2 * stdev
        return max(threshold, gap_threshold_min_ms)
 
    # Single boolean column on every original dataframe, defaulted to False so that
    # every row ends up annotated: True if its window was flagged suspicious,
    # False if it was never considered, or considered but below threshold.
    DETECTOR_COL = "detector_suspicious"
    for key, orig_df in packet_dict.items():
        if orig_df is None or orig_df.empty:
            continue
        orig_df[DETECTOR_COL] = False
 
    # Group by flow_id so each flow is inspected independently
    for flow_id, flow_df in df.groupby("flow_id"):
 
        flow_df   = flow_df.reset_index(drop=True)
        iats      = flow_df["iat_ms"].values
        dk_means  = flow_df["dk_mean_ms"].values
        dk_stdevs = flow_df["dk_stdev_ms"].values
 
        # Pre-compute the index of the next gap for every position,
        # so the burst window never bleeds into the following gap event
        next_gap = np.full(len(iats), len(iats), dtype=int)  # default: end of flow
        for k in range(len(iats) - 1, 0, -1):
            if iats[k] > gap_threshold_min_ms:
                next_gap[k] = k
            else:
                next_gap[k] = next_gap[k + 1] if k + 1 < len(iats) else len(iats)
 
        i = 1
        while i < len(iats):
 
            gap_ms   = iats[i]
            dk_mean  = dk_means[i]
            dk_stdev = dk_stdevs[i]
 
            if iats[i] > get_upper_threshold(dk_mean, dk_stdev):
 
                if dk_mean <= 0:
                    i += 1
                    continue
 
                if dk_mean <= 1:
                    n_expected = max(1, int(round(gap_ms / 1)))
                else:
                    n_expected = max(1, int(round(gap_ms / dk_mean)))
 
                j = i + 1
 
                # Hard stop: burst window cannot reach or cross the next gap
                next_gap_idx = next_gap[j] if j < len(iats) else len(iats)
                window_end   = min(j + n_expected, next_gap_idx)
 
                burst_iats = iats[j:window_end]
                burst_df   = flow_df.iloc[j:window_end]
 
                if len(burst_iats) < 1:
                    i = window_end
                    continue
 
                fast_iat_threshold_ms = get_lower_threshold(dk_mean, dk_stdev)
 
                n_fast      = int((burst_iats < fast_iat_threshold_ms).sum())
                fill_ratio  = n_fast / n_expected
 
                # detector verdict, replacing the old results.append(...) row
                is_suspicious = fill_ratio > suspicious_fill_ratio_threshold
 
                # --- write the verdict back into the original packet_dict dataframes ---
 
                if is_suspicious:
                    # mark only the rows that were actually classified as "fast"
                    # within the burst window (length n_fast, a subset of n_expected)
                    fast_mask = burst_iats < fast_iat_threshold_ms
                    fast_rows = burst_df.iloc[fast_mask]
                    for _, row in fast_rows.iterrows():
                        b_key = row["_src_key"]
                        b_idx = row["_src_idx"]
                        packet_dict[b_key].loc[b_idx, DETECTOR_COL] = True
                    # rows default to False already, so nothing to do when not suspicious
 
                i = window_end
 
            else:
                i += 1
 
    return packet_dict

### 2.2 Detector evaluation (packet-level)

In [ ]:
def _confusion_metrics(predicted, ground_truth):
    """Given two aligned boolean Series, return TP/FP/TN/FN + derived metrics."""
    tp = int((predicted & ground_truth).sum())
    fp = int((predicted & ~ground_truth).sum())
    tn = int((~predicted & ~ground_truth).sum())
    fn = int((~predicted & ground_truth).sum())
    total = tp + fp + tn + fn

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall    = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1        = (2 * precision * recall / (precision + recall)) if (precision + recall) > 0 else 0.0
    accuracy  = (tp + tn) / total if total > 0 else 0.0

    return {
        "tp": tp, "fp": fp, "tn": tn, "fn": fn, "total": total,
        "precision": precision, "recall": recall, "f1": f1, "accuracy": accuracy,
    }


def _ground_truth(df, label_cheat_types):
    if label_cheat_types is None:
        return df["is_cheating"] == True
    return (df["is_cheating"] == True) & (df["cheat_type"].isin(label_cheat_types))

In [ ]:
def evaluate_detector(
    packet_dict,
    detector_col="detector_suspicious",
    label_cheat_types=("LagSwitch",),
    per_flow=False,
    per_region=False,
    per_ip=False,
    ip_col="src_ip",
):
    """
    Compare the per-packet detector verdict (packet_dict[*][detector_col]) against
    the ground-truth label, and count TP/FP/TN/FN at the packet level.

    Ground truth for a packet is True if:
        row["is_cheating"] == True AND row["cheat_type"] in label_cheat_types
    (defaults to LagSwitch only, matching the original is_cheating_window logic;
    pass label_cheat_types=None to treat ANY is_cheating==True row as positive).

    Results are reported as:
      - "per_session": one row per key in packet_dict (i.e. per input session/capture)
      - "per_flow": one row per flow_id, across all sessions combined (only if per_flow=True)
      - "per_region": one row per region, across all sessions combined (only if per_region=True)
      - "per_ip": one row per IP (ip_col), across all sessions combined (only if per_ip=True)
      - "total": a single dict aggregating every packet across every session

    Parameters
    ----------
    packet_dict : dict[str, pd.DataFrame]
        Output of extract_suspicious_window. Each dict key is treated as one
        "session"; each dataframe must have `detector_col`, "is_cheating", and
        "cheat_type" columns.
    detector_col : str
        Column holding the boolean detector verdict per packet.
    label_cheat_types : tuple[str] or None
        Which cheat_type values count as a true positive label. None means
        "any is_cheating == True row is a positive", regardless of cheat_type.
    per_flow : bool
        If True, also return a per-flow_id breakdown (requires a "flow_id" column).
    per_region : bool
        If True, also return a per-region breakdown (requires a "region" column).
    per_ip : bool
        If True, also return a per-IP breakdown (requires an `ip_col` column,
        e.g. "src_ip"). This is packet-level TP/FP/TN/FN per IP — useful as a
        sanity check, but for player-level inference see extract_per_ip_stats,
        which returns the raw counts (flagged/total/etc.) without collapsing
        them into a verdict.
    ip_col : str
        Name of the column identifying the player/IP. Only used if per_ip=True.

    Returns
    -------
    dict with keys:
        "total"       -> dict of tp/fp/tn/fn/total/precision/recall/f1/accuracy
        "per_session" -> DataFrame, one row per session (packet_dict key)
        "per_flow"    -> DataFrame, one row per flow_id (only if per_flow=True)
        "per_region"  -> DataFrame, one row per region (only if per_region=True)
        "per_ip"      -> DataFrame, one row per IP (only if per_ip=True)
    """

    valid_items = [(key, df) for key, df in packet_dict.items() if df is not None and not df.empty]

    if not valid_items:
        print("No data available")
        return None

    sample_df = valid_items[0][1]
    if detector_col not in sample_df.columns:
        raise ValueError(
            f"Column '{detector_col}' not found — did you run extract_suspicious_window first?"
        )

    # --- per-session metrics ---
    session_rows = []
    for session_key, session_df in valid_items:
        predicted = session_df[detector_col].astype(bool)
        gt = _ground_truth(session_df, label_cheat_types)
        m = _confusion_metrics(predicted, gt)
        session_rows.append({"session": session_key, **m})

    per_session_df = pd.DataFrame(session_rows)

    # --- total metrics (all sessions combined) ---
    df = pd.concat([d for _, d in valid_items], ignore_index=True)
    predicted_total = df[detector_col].astype(bool)
    gt_total = _ground_truth(df, label_cheat_types)
    total_metrics = _confusion_metrics(predicted_total, gt_total)

    result = {
        "total": total_metrics,
        "per_session": per_session_df,
    }

    # --- optional per-flow metrics (across all sessions combined) ---
    if per_flow:
        if "flow_id" not in df.columns:
            raise ValueError("per_flow=True requires a 'flow_id' column")

        flow_rows = []
        for flow_id, flow_df in df.groupby("flow_id"):
            pred = flow_df[detector_col].astype(bool)
            gt = _ground_truth(flow_df, label_cheat_types)
            m = _confusion_metrics(pred, gt)
            flow_rows.append({"flow_id": flow_id, **m})

        result["per_flow"] = pd.DataFrame(flow_rows)

    # --- optional per-region metrics (across all sessions combined) ---
    if per_region:
        if "region" not in df.columns:
            raise ValueError("per_region=True requires a 'region' column")

        region_rows = []
        for region, region_df in df.groupby("region"):
            pred = region_df[detector_col].astype(bool)
            gt = _ground_truth(region_df, label_cheat_types)
            m = _confusion_metrics(pred, gt)
            region_rows.append({"region": region, **m})

        result["per_region"] = pd.DataFrame(region_rows)

    # --- optional per-IP metrics (across all sessions combined) ---
    if per_ip:
        if ip_col not in df.columns:
            raise ValueError(f"per_ip=True requires an '{ip_col}' column")

        ip_rows = []
        for ip, ip_df in df.groupby(ip_col):
            pred = ip_df[detector_col].astype(bool)
            gt = _ground_truth(ip_df, label_cheat_types)
            m = _confusion_metrics(pred, gt)
            ip_rows.append({ip_col: ip, **m})

        result["per_ip"] = pd.DataFrame(ip_rows)

    return result

### 2.4 Half-life sensitivity sweep

In [ ]:
def compare_half_life_impact_detection(
    pcap_dict,
    half_life_values
):
    """
    Sweep half_life values, recompute decay-based IAT features, run the
    detector, and evaluate it PER REGION (in addition to the overall total).

    Returns one row per (half_life, region) combination, plus a separate
    "overall" row per half_life aggregating all regions together — so you
    can see both the global trend and whether some regions are harder to
    detect than others.
    """
    all_results = []
    all_details = []
    for hl in half_life_values:
        print(f"Running half_life={hl}")
        processed = {}
        # recompute features per pcap
        for ts, df in pcap_dict.items():
            df_feat = decay_iat_features(df.copy(), half_life=hl)
            processed[ts] = df_feat
        detector_output_pcap = extract_suspicious_window_packet_per_packet(
            processed,
            gap_threshold_min_ms=400,
            burst_threshold_max_ms=1.0,
            suspicious_fill_ratio_threshold=0.5
        )
        results = evaluate_detector(
            detector_output_pcap,
            label_cheat_types=("LagSwitch",),
            per_region=True,
        )

        all_details.append({"half_life": hl, "results": results})

        def _row_from_metrics(m):
            tp = m.get("tp", m.get("TP", 0))
            fp = m.get("fp", m.get("FP", 0))
            tn = m.get("tn", m.get("TN", 0))
            fn = m.get("fn", m.get("FN", 0))
            alpha = tp / (tp + fn) if (tp + fn) else 0.0   # P(Detect | Cheat)
            beta  = fp / (fp + tn) if (fp + tn) else 0.0   # P(Detect | No Cheat)
            return tp, fp, tn, fn, alpha, beta

        # --- one row per region for this half_life ---
        per_region_df = results.get("per_region")
        if per_region_df is not None:
            for _, region_row in per_region_df.iterrows():
                tp, fp, tn, fn, alpha, beta = _row_from_metrics(region_row)
                all_results.append({
                    "half_life": hl,
                    "region": region_row["region"],
                    "TP": tp,
                    "FP": fp,
                    "TN": tn,
                    "FN": fn,
                    "P(Detect|Cheat)": alpha,
                    "P(Detect|NoCheat)": beta,
                })

        # --- overall row (all regions combined) for this half_life ---
        tp, fp, tn, fn, alpha, beta = _row_from_metrics(results.get("total", {}))
        all_results.append({
            "half_life": hl,
            "region": "overall",
            "TP": tp,
            "FP": fp,
            "TN": tn,
            "FN": fn,
            "P(Detect|Cheat)": alpha,
            "P(Detect|NoCheat)": beta,
        })

    return pd.DataFrame(all_results), all_details

### 2.5 Per-player inference via likelihood ratios (LLR)

Aggregate the packet-level detector output per player/IP, then use a mixture-model log-likelihood-ratio test to decide, per player, whether they cheated at some point in the session.

In [ ]:
def extract_per_ip_stats(
    packet_dict,
    detector_col="detector_suspicious",
    label_cheat_types=("LagSwitch",),
    ip_col="src_ip",
):
    """
    Aggregate packet-level detector output per IP (player), returning the raw
    counts needed as input to a downstream per-player decision rule (e.g. a
    log-likelihood-ratio test) rather than a collapsed verdict.

    For each IP, this reports:
      - n_packets        : total packets considered for this IP across all sessions
      - n_flagged         : count of packets where detector_col == True
      - flagged_rate      : n_flagged / n_packets
      - n_cheat_label     : count of packets where ground truth is positive
                             (is_cheating & cheat_type in label_cheat_types)
      - cheat_label_rate  : n_cheat_label / n_packets
      - n_flagged_and_cheat : overlap (packets that are both flagged and truly
                               labeled cheating) — useful as an evidence count
      - n_sessions        : number of distinct sessions (packet_dict keys) this
                             IP appears in
      - n_flows           : number of distinct flow_ids this IP appears in
                             (only included if a "flow_id" column exists)
      - any_cheat_label   : True if this IP has at least one ground-truth
                             positive packet anywhere (useful as a simple
                             "did this player ever actually cheat" reference
                             label, separate from whatever decision rule you
                             apply to the detector's flags)

    This intentionally does NOT threshold n_flagged into a per-player verdict;
    that decision (e.g. via an LLR test) is left to the caller.

    Parameters
    ----------
    packet_dict : dict[str, pd.DataFrame]
        Output of extract_suspicious_window. Each dataframe must have
        `detector_col`, "is_cheating", "cheat_type", and `ip_col` columns.
    detector_col : str
        Column holding the boolean detector verdict per packet.
    label_cheat_types : tuple[str] or None
        Which cheat_type values count as a ground-truth positive packet.
        None means any is_cheating==True row counts.
    ip_col : str
        Name of the column identifying the player/IP.

    Returns
    -------
    pd.DataFrame, one row per IP, sorted by n_flagged descending.
    """

    valid_items = [(key, df) for key, df in packet_dict.items() if df is not None and not df.empty]

    if not valid_items:
        print("No data available")
        return None

    sample_df = valid_items[0][1]
    if detector_col not in sample_df.columns:
        raise ValueError(
            f"Column '{detector_col}' not found — did you run extract_suspicious_window first?"
        )
    if ip_col not in sample_df.columns:
        raise ValueError(f"Column '{ip_col}' not found in packet dataframes")

    has_flow_id = "flow_id" in sample_df.columns
    has_region = "region" in sample_df.columns

    # keep session key alongside each row so we can count distinct sessions per IP
    frames = []
    for session_key, session_df in valid_items:
        tmp = session_df.copy()
        tmp["_session_key"] = session_key
        frames.append(tmp)

    df = pd.concat(frames, ignore_index=True)

    df["_ground_truth"] = _ground_truth(df, label_cheat_types)
    df["_flagged"] = df[detector_col].astype(bool)

    rows = []
    for ip, ip_df in df.groupby(ip_col):
        n_packets = len(ip_df)
        n_flagged = int(ip_df["_flagged"].sum())
        n_cheat_label = int(ip_df["_ground_truth"].sum())
        n_flagged_and_cheat = int((ip_df["_flagged"] & ip_df["_ground_truth"]).sum())
        n_sessions = ip_df["_session_key"].nunique()

        row = {
            ip_col: ip,
            "n_packets": n_packets,
            "n_flagged": n_flagged,
            "flagged_rate": n_flagged / n_packets if n_packets > 0 else 0.0,
            "n_cheat_label": n_cheat_label,
            "cheat_label_rate": n_cheat_label / n_packets if n_packets > 0 else 0.0,
            "n_flagged_and_cheat": n_flagged_and_cheat,
            "n_sessions": n_sessions,
            "any_cheat_label": bool(n_cheat_label > 0),
        }

        if has_flow_id:
            row["n_flows"] = ip_df["flow_id"].nunique()

        if has_region:
            # each IP is assumed to belong to exactly one region; if that's
            # ever violated, n_regions > 1 will reveal the inconsistency
            regions_seen = ip_df["region"].dropna().unique()
            row["region"] = regions_seen[0] if len(regions_seen) > 0 else None
            row["n_regions"] = len(regions_seen)

        rows.append(row)

    result_df = pd.DataFrame(rows).sort_values("n_flagged", ascending=False).reset_index(drop=True)
    return result_df

In [ ]:
def compute_llr_rates_from_confusion(total_metrics):
    """
    Derive the binomial rates p0 (innocent baseline flag rate) and p1 (cheating
    flag rate) directly from a packet-level confusion matrix dict, e.g. the
    "total" entry returned by evaluate_detector.

        p0 = P(flagged | not actually cheating)  = fp / (fp + tn)
        p1 = P(flagged | actually cheating)      = tp / (tp + fn)   (== recall)

    Parameters
    ----------
    total_metrics : dict
        Must contain keys "tp", "fp", "tn", "fn" (as returned by
        evaluate_detector(...)["total"]).

    Returns
    -------
    (p0, p1) tuple of floats.
    """
    tp = total_metrics["tp"]
    fp = total_metrics["fp"]
    tn = total_metrics["tn"]
    fn = total_metrics["fn"]

    p0 = fp / (fp + tn) if (fp + tn) > 0 else 0.0
    p1 = tp / (tp + fn) if (tp + fn) > 0 else 0.0

    if not (0 < p0 < p1 < 1):
        raise ValueError(
            f"Expected 0 < p0 < p1 < 1 for a valid LLR test, got p0={p0:.6f}, p1={p1:.6f}. "
            "Check your confusion matrix — if p1 <= p0, the detector isn't discriminative "
            "enough at the packet level for this test to make sense."
        )

    return p0, p1

In [ ]:
def compute_player_llr(per_ip_stats, p0, p1, n_col="n_packets", k_col="n_flagged"):
    """
    Compute a per-player log-likelihood-ratio (LLR) using a two-component
    MIXTURE model, rather than assuming the whole session is homogeneously
    "cheating" or "innocent".

    Why a mixture: cheating happens in bursts/episodes, not continuously
    across the whole game. A fixed-p1 binomial test (treating every packet as
    drawn from Binomial(n, p1) if cheating) forces an enormous, unrealistic
    flag rate requirement as n grows (e.g. needing ~9% of ALL packets flagged
    for a 100k-packet session just to reach LLR=0), so it fails for any real
    player and produces uniformly large negative LLRs dominated by n rather
    than by evidence of cheating.

    Mixture model: for a player with n packets, assume an unknown fraction pi
    of their session was spent cheating (per-packet flag rate p1) and the
    rest was normal play (flag rate p0). The number of flagged packets k is
    modeled as:

        k ~ Binomial(n, pi*p1 + (1-pi)*p0)

    This is a single binomial with an EFFECTIVE rate that interpolates
    between p0 (pi=0, fully innocent) and p1 (pi=1, cheating the entire
    session). We find the maximum-likelihood pi_hat in [0, 1] (closed form,
    since the effective rate is linear in pi: just clip the observed rate
    k/n into [p0, p1] and invert), then compute the likelihood-ratio test
    statistic comparing that fit against the pi=0 (pure innocent) model:

        LLR = 2 * (log L(pi_hat) - log L(pi=0))

    By Wilks' theorem this is asymptotically chi-squared with 1 degree of
    freedom under H0 (pi=0), which also gives a principled way to pick a
    threshold (e.g. tau = chi2.ppf(0.95, df=1) for a ~5% player-level false
    alarm rate) in addition to the empirical sweep in sweep_llr_threshold.

    Unlike the fixed-p1 version, LLR=0 whenever the observed flag rate is at
    or below the baseline p0, regardless of how large n is — so heavy
    players are no longer penalized just for having many packets.

    Parameters
    ----------
    per_ip_stats : pd.DataFrame
        Output of extract_per_ip_stats (or any dataframe with n_col/k_col).
    p0, p1 : float
        Innocent / cheating per-packet flag rates, e.g. from
        compute_llr_rates_from_confusion. Must satisfy 0 < p0 < p1 < 1.
    n_col, k_col : str
        Column names for total packets and flagged-packet counts per player.

    Returns
    -------
    per_ip_stats with two added columns (does not mutate the input):
        "pi_hat" : estimated fraction of the session spent cheating, in [0, 1]
        "llr"    : the likelihood-ratio test statistic (>= 0), large values
                   favor "this player cheated for some fraction of their
                   session" over "this player never cheated"
    """
    if not (0 < p0 < p1 < 1):
        raise ValueError(f"Expected 0 < p0 < p1 < 1, got p0={p0}, p1={p1}")

    df = per_ip_stats.copy()

    n = df[n_col].astype(float).values
    k = df[k_col].astype(float).values

    with np.errstate(divide="ignore", invalid="ignore"):
        observed_rate = np.where(n > 0, k / n, 0.0)

    # closed-form MLE: effective rate is linear in pi, so just clip the
    # observed rate into the achievable range [p0, p1] and invert
    pi_hat = (observed_rate - p0) / (p1 - p0)
    pi_hat = np.clip(pi_hat, 0.0, 1.0)

    def _binom_loglik(pi, n, k):
        rate = pi * p1 + (1 - pi) * p0
        rate = np.clip(rate, 1e-12, 1 - 1e-12)
        # binomial coefficient term cancels in the likelihood RATIO, so it's
        # omitted here for both H1 and H0 — safe since LLR is a difference
        return k * np.log(rate) + (n - k) * np.log(1 - rate)

    ll_h1 = _binom_loglik(pi_hat, n, k)
    ll_h0 = _binom_loglik(0.0, n, k)

    llr = 2 * (ll_h1 - ll_h0)
    llr = np.clip(llr, 0.0, None)  # guard tiny negative values from floating point noise

    df["pi_hat"] = pi_hat
    df["llr"] = llr
    return df

In [ ]:
def sweep_llr_threshold(
    per_ip_stats_with_llr,
    label_col="any_cheat_label",
    n_thresholds=200,
):
    """
    Sweep the LLR decision threshold tau across the range of observed LLR
    values, and compute player-level TP/FP/TN/FN/precision/recall/f1 at each
    tau against the known label column (e.g. "any_cheat_label" from
    extract_per_ip_stats). Use this to pick tau and to plot a player-level
    ROC / precision-recall curve.

    A player is classified "cheater" at threshold tau if llr > tau.

    Parameters
    ----------
    per_ip_stats_with_llr : pd.DataFrame
        Output of compute_player_llr (must have an "llr" column).
    label_col : str
        Boolean column holding the ground-truth player-level label.
    n_thresholds : int
        Number of threshold values to evaluate, evenly spaced across the
        observed LLR range.

    Returns
    -------
    pd.DataFrame, one row per tau, with columns:
        tau, tp, fp, tn, fn, precision, recall, f1, accuracy,
        fpr (false positive rate), tpr (true positive rate, == recall)
    """
    df = per_ip_stats_with_llr
    if "llr" not in df.columns:
        raise ValueError("per_ip_stats_with_llr must have an 'llr' column — run compute_player_llr first")
    if label_col not in df.columns:
        raise ValueError(f"Column '{label_col}' not found in per_ip_stats_with_llr")

    llr_values = df["llr"].values
    gt = df[label_col].astype(bool).values

    lo, hi = llr_values.min(), llr_values.max()
    # pad slightly so the extreme ends (all-cheater / all-innocent) are included
    pad = (hi - lo) * 0.01 if hi > lo else 1.0
    taus = np.linspace(lo - pad, hi + pad, n_thresholds)

    rows = []
    for tau in taus:
        predicted = llr_values > tau

        tp = int((predicted & gt).sum())
        fp = int((predicted & ~gt).sum())
        tn = int((~predicted & ~gt).sum())
        fn = int((~predicted & gt).sum())

        precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        recall    = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        f1        = (2 * precision * recall / (precision + recall)) if (precision + recall) > 0 else 0.0
        accuracy  = (tp + tn) / len(gt) if len(gt) > 0 else 0.0
        fpr       = fp / (fp + tn) if (fp + tn) > 0 else 0.0

        rows.append({
            "tau": tau, "tp": tp, "fp": fp, "tn": tn, "fn": fn,
            "precision": precision, "recall": recall, "f1": f1,
            "accuracy": accuracy, "fpr": fpr, "tpr": recall,
        })

    return pd.DataFrame(rows)

In [ ]:
def plot_llr_roc(threshold_sweep_df):
    """
    Plot the player-level ROC curve (TPR vs FPR) from sweep_llr_threshold's
    output, and mark the point maximizing F1 as a candidate operating point.

    Returns the matplotlib (fig, ax) tuple.
    """
    fig, ax = plt.subplots(figsize=(6, 6))

    ax.plot(threshold_sweep_df["fpr"], threshold_sweep_df["tpr"], color="steelblue", linewidth=2)
    ax.plot([0, 1], [0, 1], color="gray", linestyle="--", linewidth=1, alpha=0.6, label="Random guess")

    best_idx = threshold_sweep_df["f1"].idxmax()
    best_row = threshold_sweep_df.loc[best_idx]
    ax.scatter(
        [best_row["fpr"]], [best_row["tpr"]],
        color="red", s=80, zorder=5,
        label=f"Best F1 @ tau={best_row['tau']:.2f} (P={best_row['precision']:.2f}, R={best_row['recall']:.2f})",
    )

    ax.set_xlabel("False Positive Rate (player-level)")
    ax.set_ylabel("True Positive Rate / Recall (player-level)")
    ax.set_title("Player-level LLR ROC curve")
    ax.legend(loc="lower right")
    ax.grid(True, alpha=0.3)
    fig.tight_layout()

    return fig, ax

In [ ]:
def plot_player_llr_scatter(
    per_ip_stats_with_llr,
    label_col="any_cheat_label",
    x_col="n_packets",
    log_x=True,
    title=None,
    ax=None,
):
    """
    Scatter plot of each player's LLR (y-axis) against x_col (default
    n_packets, x-axis), colored by ground-truth label. This is the main
    sanity-check plot for the mixture-model LLR: it should show True-labeled
    players (actual cheaters) sitting at visibly higher LLR than False-labeled
    players, with NO systematic trend across n_packets (the old fixed-p1 LLR
    failed this exact check — its score was dominated by n rather than guilt).

    Parameters
    ----------
    per_ip_stats_with_llr : pd.DataFrame
        Output of compute_player_llr or compute_player_llr_by_region (must
        have an "llr" column).
    label_col : str
        Boolean column holding the ground-truth player-level label.
    x_col : str
        Column to use as the x-axis (default "n_packets", to visually check
        for any remaining n-dependence).
    log_x : bool
        Use a log scale on the x-axis (recommended when x_col spans a wide
        range, e.g. n_packets).
    title : str or None
        Plot title. If None, a sensible default is used.
    ax : matplotlib Axes or None
        If provided, plot onto this axes instead of creating a new figure
        (used by plot_player_llr_scatter_by_region to facet into subplots).

    Returns
    -------
    The matplotlib (fig, ax) tuple (fig is None if ax was passed in).
    """
    df = per_ip_stats_with_llr
    if "llr" not in df.columns:
        raise ValueError("per_ip_stats_with_llr must have an 'llr' column")
    if label_col not in df.columns:
        raise ValueError(f"Column '{label_col}' not found")

    fig = None
    if ax is None:
        fig, ax = plt.subplots(figsize=(8, 6))

    gt = df[label_col].astype(bool)

    ax.scatter(
        df.loc[~gt, x_col], df.loc[~gt, "llr"],
        color="steelblue", alpha=0.6, s=30, label="Innocent (label=False)", zorder=2,
    )
    ax.scatter(
        df.loc[gt, x_col], df.loc[gt, "llr"],
        color="red", alpha=0.7, s=40, label="Cheater (label=True)", zorder=3,
    )

    if log_x:
        ax.set_xscale("log")

    ax.set_xlabel(x_col)
    ax.set_ylabel("LLR")
    ax.set_title(title or "Player LLR vs. " + x_col)
    ax.legend(loc="best")
    ax.grid(True, alpha=0.3)

    if fig is not None:
        fig.tight_layout()

    return fig, ax

In [ ]:
def plot_player_llr_scatter_by_region(
    per_ip_stats_with_llr,
    region_col="region",
    label_col="any_cheat_label",
    x_col="n_packets",
    log_x=True,
    ncols=2,
):
    """
    Facet plot_player_llr_scatter into one subplot per region, sharing the
    same y-axis (LLR) scale across subplots so regions are visually
    comparable. Use this when LLR rates were calibrated per region (via
    compute_player_llr_by_region) to check separation quality independently
    in each region.

    Parameters
    ----------
    per_ip_stats_with_llr : pd.DataFrame
        Output of compute_player_llr_by_region (must have "llr" and
        `region_col` columns).
    region_col : str
        Column identifying each player's region.
    label_col, x_col, log_x : see plot_player_llr_scatter.
    ncols : int
        Number of subplot columns in the grid.

    Returns
    -------
    The matplotlib (fig, axes) tuple.
    """
    if region_col not in per_ip_stats_with_llr.columns:
        raise ValueError(f"Column '{region_col}' not found in per_ip_stats_with_llr")

    regions = sorted(per_ip_stats_with_llr[region_col].dropna().unique(), key=str)
    n_regions = len(regions)
    nrows = int(np.ceil(n_regions / ncols))

    fig, axes = plt.subplots(nrows, ncols, figsize=(6 * ncols, 5 * nrows), squeeze=False)
    axes_flat = axes.flatten()

    y_min = per_ip_stats_with_llr["llr"].min()
    y_max = per_ip_stats_with_llr["llr"].max()
    y_pad = (y_max - y_min) * 0.05 if y_max > y_min else 1.0

    for idx, region in enumerate(regions):
        region_df = per_ip_stats_with_llr[per_ip_stats_with_llr[region_col] == region]
        plot_player_llr_scatter(
            region_df,
            label_col=label_col,
            x_col=x_col,
            log_x=log_x,
            title=f"region={region}",
            ax=axes_flat[idx],
        )
        axes_flat[idx].set_ylim(y_min - y_pad, y_max + y_pad)

    # hide any unused subplot slots
    for idx in range(n_regions, len(axes_flat)):
        axes_flat[idx].set_visible(False)

    fig.suptitle("Player LLR by region", fontsize=14)
    fig.tight_layout()

    return fig, axes

In [ ]:
def sweep_llr_threshold_by_region(
    per_ip_stats_with_llr,
    region_col="region",
    label_col="any_cheat_label",
    n_thresholds=200,
):
    """
    Like sweep_llr_threshold, but run independently within each region and
    return one ROC sweep DataFrame per region. Use this if regions have
    enough labeled players to justify a region-specific tau, instead of one
    global tau applied across all regions' (differently-calibrated) LLRs.

    Returns
    -------
    dict[region -> DataFrame], each in the same shape as sweep_llr_threshold's
    output.
    """
    if region_col not in per_ip_stats_with_llr.columns:
        raise ValueError(f"Column '{region_col}' not found in per_ip_stats_with_llr")

    sweeps_by_region = {}
    for region, region_df in per_ip_stats_with_llr.groupby(region_col):
        sweeps_by_region[region] = sweep_llr_threshold(
            region_df, label_col=label_col, n_thresholds=n_thresholds
        )

    return sweeps_by_region

In [ ]:
def plot_llr_roc_by_region(sweeps_by_region, ncols=2):
    """
    Facet plot_llr_roc into one subplot per region, given the dict returned
    by sweep_llr_threshold_by_region.

    Returns
    -------
    The matplotlib (fig, axes) tuple.
    """
    regions = sorted(sweeps_by_region.keys(), key=str)
    n_regions = len(regions)
    nrows = int(np.ceil(n_regions / ncols))

    fig, axes = plt.subplots(nrows, ncols, figsize=(6 * ncols, 6 * nrows), squeeze=False)
    axes_flat = axes.flatten()

    for idx, region in enumerate(regions):
        sweep_df = sweeps_by_region[region]
        ax = axes_flat[idx]

        ax.plot(sweep_df["fpr"], sweep_df["tpr"], color="steelblue", linewidth=2)
        ax.plot([0, 1], [0, 1], color="gray", linestyle="--", linewidth=1, alpha=0.6, label="Random guess")

        best_idx = sweep_df["f1"].idxmax()
        best_row = sweep_df.loc[best_idx]
        ax.scatter(
            [best_row["fpr"]], [best_row["tpr"]],
            color="red", s=80, zorder=5,
            label=f"Best F1 @ tau={best_row['tau']:.2f}",
        )

        ax.set_xlabel("FPR")
        ax.set_ylabel("TPR")
        ax.set_title(f"region={region}")
        ax.legend(loc="lower right", fontsize=8)
        ax.grid(True, alpha=0.3)

    for idx in range(n_regions, len(axes_flat)):
        axes_flat[idx].set_visible(False)

    fig.suptitle("Player-level LLR ROC by region", fontsize=14)
    fig.tight_layout()

    return fig, axes

### 2.8 Gap-event-based evaluation

A middle ground between packet-level and fixed windows: anchor each "event" to an actual IAT gap (the gap packet plus the next `n_following` packets), and score detection at that event level.

In [ ]:
def evaluate_detector_gap_events(
    packet_dict,
    gap_threshold_ms=400,
    n_following=10,
    detector_col="detector_suspicious",
    label_cheat_types=("LagSwitch",),
    timestamp_col="timestamp",
    per_flow=False,
    per_region=False,
    per_ip=False,
    ip_col="src_ip",
    per_isp=False,
    isp_col="isp",
):
    """
    Gap-event-level evaluation: instead of fixed-size time windows
    (evaluate_detector_windowed) or raw packets (evaluate_detector), this
    anchors each "event" to an actual IAT gap (the same trigger condition
    the detector itself uses), then asks whether the detector flagged
    anything inside that event window.

    Event construction
    -------------------
    For every packet in a flow whose IAT exceeds `gap_threshold_ms`, that
    packet starts a new event. The event's window is:
        [gap packet, gap packet + 1, ..., gap packet + n_following]
    i.e. the gap packet itself plus the next `n_following` packets in the
    same flow (fewer if the flow ends first). Every qualifying gap spawns
    its OWN independent event — overlapping windows from nearby gaps are
    NOT merged, so a packet may belong to more than one event.

    Event labeling (ground truth)
    ------------------------------
    - LagSwitch  : at least one packet in the event window has
                   is_cheating=True and cheat_type in label_cheat_types
    - Honest     : otherwise — this includes windows with NO cheating at all,
                   AND windows whose only cheating is a DIFFERENT type (e.g.
                   FixedDelay, DOS) with no LagSwitch packet present. Per
                   spec, "honest" here means "not labeled LagSwitch", not
                   "no cheating whatsoever".

    Event verdict (detector)
    -------------------------
    - Detected   : at least one packet in the event window has
                   detector_col == True
    - Missed     : no packet in the event window was flagged

    TP/FP/TN/FN are then counted at the EVENT level: TP = LagSwitch event
    that was detected, FN = LagSwitch event that was missed, FP = honest
    event that was (incorrectly) detected, TN = honest event correctly left
    undetected.

    Parameters
    ----------
    packet_dict : dict[str, pd.DataFrame]
        Output of extract_suspicious_window. Each dataframe must have
        `detector_col`, "is_cheating", "cheat_type", "flow_id", "iat", and
        `timestamp_col` columns.
    gap_threshold_ms : float
        IAT threshold (ms) defining a gap / event trigger.
    n_following : int
        Number of packets after the gap packet to include in the event
        window (the gap packet itself is always included).
    detector_col : str
        Column holding the boolean detector verdict per packet.
    label_cheat_types : tuple[str] or None
        Which cheat_type values count as "LagSwitch" for event labeling.
        None means any is_cheating==True row counts as the positive type
        (rarely what you want here, but kept for consistency with the other
        evaluate_* functions).
    timestamp_col : str
        Column holding each packet's absolute timestamp; used only to sort
        packets into the correct order within each flow before scanning for
        gaps and following packets.
    per_flow, per_region, per_ip, ip_col :
        Same semantics as evaluate_detector / evaluate_detector_windowed,
        but breakdowns are now over EVENTS rather than packets or time
        windows.
    per_isp : bool
        If True, also return a per-ISP breakdown (requires an `isp_col`
        column, e.g. "isp"). Same event-level TP/FP/TN/FN treatment as
        per_region/per_ip, just grouped on ISP instead.
    isp_col : str
        Name of the column identifying the ISP. Only used if per_isp=True.

    Returns
    -------
    dict with the same shape as evaluate_detector's return value (keys:
    "total", "per_session", and optionally "per_flow"/"per_region"/"per_ip"),
    except all counts are EVENT counts. An additional "events" key holds the
    full per-event table (one row per gap event) with its labeled/detected
    booleans and metadata, including:
        - gap_size     : the triggering gap packet's IAT, in ms
        - dk_mean_filt : the triggering gap packet's dk_iat_mean_filt value
                          (raw, same units as in the source dataframe)
        - dd_filt       : the triggering gap packet's dk_iat_std_filt value
                          (raw, same units as in the source dataframe)
    useful for further inspection or custom aggregation (e.g. checking
    whether missed events cluster at small gap_size or unusual dk_mean_filt).
    """

    valid_items = [(key, df) for key, df in packet_dict.items() if df is not None and not df.empty]

    if not valid_items:
        print("No data available")
        return None

    sample_df = valid_items[0][1]
    for required_col in ("flow_id", "iat", detector_col, timestamp_col):
        if required_col not in sample_df.columns:
            raise ValueError(f"Column '{required_col}' not found in packet dataframes")

    has_region = "region" in sample_df.columns
    has_ip = ip_col in sample_df.columns
    has_isp = isp_col in sample_df.columns
    has_dk_mean = "dk_iat_mean_filt" in sample_df.columns
    has_dk_std = "dk_iat_std_filt" in sample_df.columns

    # --- build the per-event table ---
    event_rows = []
    for session_key, session_df in valid_items:
        gt_mask = _ground_truth(session_df, label_cheat_types)
        flagged_mask = session_df[detector_col].astype(bool)

        for flow_id, flow_df in session_df.groupby("flow_id"):
            flow_df = flow_df.sort_values(timestamp_col)
            flow_idx = flow_df.index.to_numpy()
            n_rows = len(flow_idx)

            iat_ms = flow_df["iat"].values * 1000
            gap_positions = np.nonzero(iat_ms > gap_threshold_ms)[0]

            if len(gap_positions) == 0:
                continue

            flow_gt = gt_mask.loc[flow_idx].values
            flow_flagged = flagged_mask.loc[flow_idx].values
            flow_region = flow_df["region"].values if has_region else None
            flow_ip = flow_df[ip_col].values if has_ip else None
            flow_isp = flow_df[isp_col].values if has_isp else None
            flow_dk_mean = flow_df["dk_iat_mean_filt"].values if has_dk_mean else None
            flow_dk_std = flow_df["dk_iat_std_filt"].values if has_dk_std else None

            for pos in gap_positions:
                window_end = min(pos + n_following + 1, n_rows)  # +1 to include the gap packet itself
                window_labeled = bool(flow_gt[pos:window_end].any())
                window_detected = bool(flow_flagged[pos:window_end].any())

                row = {
                    "session": session_key,
                    "flow_id": flow_id,
                    "gap_position": pos,
                    "window_size": window_end - pos,
                    "gap_size": iat_ms[pos],
                    "ground_truth": window_labeled,
                    "flagged": window_detected,
                }
                if has_region:
                    row["region"] = flow_region[pos]
                if has_ip:
                    row[ip_col] = flow_ip[pos]
                if has_isp:
                    row[isp_col] = flow_isp[pos]
                if has_dk_mean:
                    row["dk_mean_filt"] = flow_dk_mean[pos]
                if has_dk_std:
                    row["dd_filt"] = flow_dk_std[pos]

                event_rows.append(row)

    if not event_rows:
        print("No gap events found above threshold")
        return None

    events_df = pd.DataFrame(event_rows)

    # --- per-session metrics (over events) ---
    session_rows = []
    for session_key, session_events in events_df.groupby("session"):
        m = _confusion_metrics(session_events["flagged"], session_events["ground_truth"])
        session_rows.append({"session": session_key, **m})

    per_session_df = pd.DataFrame(session_rows)

    # --- total metrics (all events combined) ---
    total_metrics = _confusion_metrics(events_df["flagged"], events_df["ground_truth"])

    result = {
        "total": total_metrics,
        "per_session": per_session_df,
        "events": events_df,
    }

    # --- optional per-flow metrics (over events, across all sessions) ---
    if per_flow:
        flow_rows = []
        for flow_id, flow_events in events_df.groupby("flow_id"):
            m = _confusion_metrics(flow_events["flagged"], flow_events["ground_truth"])
            flow_rows.append({"flow_id": flow_id, **m})
        result["per_flow"] = pd.DataFrame(flow_rows)

    # --- optional per-region metrics (over events, across all sessions) ---
    if per_region:
        if not has_region:
            raise ValueError("per_region=True requires a 'region' column")

        region_rows = []
        for region, region_events in events_df.groupby("region"):
            m = _confusion_metrics(region_events["flagged"], region_events["ground_truth"])
            region_rows.append({"region": region, **m})
        result["per_region"] = pd.DataFrame(region_rows)

    # --- optional per-IP metrics (over events, across all sessions) ---
    if per_ip:
        if not has_ip:
            raise ValueError(f"per_ip=True requires an '{ip_col}' column")

        ip_rows = []
        for ip, ip_events in events_df.groupby(ip_col):
            m = _confusion_metrics(ip_events["flagged"], ip_events["ground_truth"])
            ip_rows.append({ip_col: ip, **m})
        result["per_ip"] = pd.DataFrame(ip_rows)

    # --- optional per-ISP metrics (over events, across all sessions) ---
    if per_isp:
        if not has_isp:
            raise ValueError(f"per_isp=True requires an '{isp_col}' column")

        isp_rows = []
        for isp, isp_events in events_df.groupby(isp_col):
            m = _confusion_metrics(isp_events["flagged"], isp_events["ground_truth"])
            row = {isp_col: isp, **m}
            # diagnostic: how many distinct IPs are aggregated into this ISP's
            # row, so it's clear when an ISP with very few events is really
            # just a single-player ISP rather than a grouping mistake
            if has_ip:
                row["n_ips"] = isp_events[ip_col].nunique()
            isp_rows.append(row)
        result["per_isp"] = pd.DataFrame(isp_rows)

    return result

In [ ]:
def compare_half_life_impact_detection_gap_events_per_isp(
    pcap_dict,
    half_life_values,
    gap_threshold_min_ms=400,
    burst_threshold_max_ms=1.0,
    suspicious_fill_ratio_threshold=0.5,
    gap_threshold_ms=400,
    n_following=10,
    label_cheat_types=("LagSwitch",),
    isp_col="isp",
):
    """
    Sweep half_life values, recompute decay-based IAT features, run the
    detector, and evaluate it with evaluate_detector_gap_events PER ISP.

    Returns one row per (half_life, isp) combination (event-level TP/FP/TN/FN
    + precision/recall/f1/accuracy), plus a helper to find the best half_life
    per ISP by F1 (see find_best_half_life_per_isp).

    Parameters
    ----------
    pcap_dict : dict[str, pd.DataFrame]
        Same input shape as compare_half_life_impact_detection.
    half_life_values : iterable
        half_life values to sweep.
    gap_threshold_min_ms, burst_threshold_max_ms, suspicious_fill_ratio_threshold :
        Passed through to extract_suspicious_window_packet_per_packet (the
        detector itself).
    gap_threshold_ms, n_following, label_cheat_types, isp_col :
        Passed through to evaluate_detector_gap_events (the gap-event
        evaluation methodology).

    Returns
    -------
    (pd.DataFrame, list) tuple:
        - DataFrame with one row per (half_life, isp), including "n_ips"
          (distinct IPs aggregated into that ISP's row at that half_life)
        - all_details: list of {"half_life": hl, "results": <full dict
          returned by evaluate_detector_gap_events, including "events">}
          for deeper inspection
    """
    all_results = []
    all_details = []

    for hl in half_life_values:
        print(f"Running half_life={hl}")
        processed = {}
        for ts, df in pcap_dict.items():
            df_feat = decay_iat_features(df.copy(), half_life=hl)
            processed[ts] = df_feat

        detector_output_pcap = extract_suspicious_window_packet_per_packet(
            processed,
            gap_threshold_min_ms=gap_threshold_min_ms,
            burst_threshold_max_ms=burst_threshold_max_ms,
            suspicious_fill_ratio_threshold=suspicious_fill_ratio_threshold,
        )

        results = evaluate_detector_gap_events(
            detector_output_pcap,
            gap_threshold_ms=gap_threshold_ms,
            n_following=n_following,
            label_cheat_types=label_cheat_types,
            per_isp=True,
            isp_col=isp_col,
        )

        all_details.append({"half_life": hl, "results": results})

        if results is None:
            continue

        per_isp_df = results.get("per_isp")
        if per_isp_df is None:
            continue

        for _, isp_row in per_isp_df.iterrows():
            row = {"half_life": hl, isp_col: isp_row[isp_col]}
            for col in ("tp", "fp", "tn", "fn", "precision", "recall", "f1", "accuracy", "n_ips"):
                if col in isp_row:
                    row[col] = isp_row[col]
            all_results.append(row)

    return pd.DataFrame(all_results), all_details

In [ ]:
def find_best_half_life_per_isp(
    per_isp_half_life_df,
    metric="f1",
    isp_col="isp",
):
    """
    For each ISP:
      - If positive events exist, maximize `metric` (default: f1).
      - If no positive events exist, minimize FP.

    Returns one row per ISP.
    """
    if metric not in per_isp_half_life_df.columns:
        raise ValueError(f"Column '{metric}' not found in per_isp_half_life_df")

    best_rows = []

    for isp, isp_df in per_isp_half_life_df.groupby(isp_col):

        # Determine whether this ISP has any positive labels
        has_positive_events = False
        if {"tp", "fn"}.issubset(isp_df.columns):
            has_positive_events = (isp_df["tp"] + isp_df["fn"]).max() > 0

        if has_positive_events:
            best_idx = isp_df[metric].idxmax()
        else:
            if "fp" not in isp_df.columns:
                raise ValueError(
                    "No positive events for ISP, but 'fp' column is missing."
                )

            # Lowest FP wins; highest TN breaks ties if available
            sort_cols = ["fp"]
            ascending = [True]

            if "tn" in isp_df.columns:
                sort_cols.append("tn")
                ascending.append(False)

            best_idx = (
                isp_df.sort_values(sort_cols, ascending=ascending)
                .index[0]
            )

        best_row = isp_df.loc[best_idx].to_dict()
        best_row["best_half_life"] = best_row.pop("half_life")
        best_rows.append(best_row)

    result_df = pd.DataFrame(best_rows)

    ordered_cols = [isp_col, "best_half_life", metric]
    remaining_cols = [c for c in result_df.columns if c not in ordered_cols]
    result_df = result_df[ordered_cols + remaining_cols]

    return result_df.sort_values(metric, ascending=False).reset_index(drop=True)

### 2.9 Baseline detectors

In [ ]:
def extract_suspicious_window_fixed_threshold(
    packet_dict,
    gap_threshold_ms=400,
    remove_fd=False,
    remove_dos=False,
    detector_col="detector_suspicious_fixed",
):
    """
    Baseline lag-switch detector using a single fixed IAT threshold instead of
    the adaptive dk_mean/dk_stdev statistics. Flags every packet whose IAT
    exceeds `gap_threshold_ms` directly — no burst-window / fill-ratio logic,
    just "is this gap itself too big".

    Adds a boolean column (`detector_col`) to every dataframe in packet_dict:
    True for packets where iat_ms > gap_threshold_ms, False otherwise
    (including rows filtered out by phase/NaN/remove_fd/remove_dos, which are
    never considered and default to False).

    Parameters
    ----------
    packet_dict : dict[str, pd.DataFrame]
        Same shape as expected by extract_suspicious_window.
    gap_threshold_ms : float
        Fixed IAT threshold (ms). Any packet with iat_ms > gap_threshold_ms
        is flagged as suspicious.
    remove_fd : bool
        If True, drop rows that are FixedDelay-cheating before evaluating
        (mirrors extract_suspicious_window's remove_fd).
    remove_dos : bool
        If True, drop rows that are DOS-cheating before evaluating.
    detector_col : str
        Name of the boolean output column to write back into packet_dict.

    Returns
    -------
    packet_dict, mutated in place with `detector_col` added to every dataframe.
    """

    all_df = []
    source_keys = []
    source_orig_idx = []

    for key, df in packet_dict.items():
        if df is not None and not df.empty:
            all_df.append(df)
            source_keys.extend([key] * len(df))
            source_orig_idx.extend(df.index.tolist())

    if not all_df:
        print("No data available")
        return packet_dict

    df = pd.concat(all_df, ignore_index=True)
    df["_src_key"] = source_keys
    df["_src_idx"] = source_orig_idx

    mask = (
        (df["phase"] == "playing") &
        (df["iat"].notna()) &
        (df["region"].notna())
    )

    if remove_fd:
        mask &= (df["is_cheating"] == False) | (df["cheat_type"] != "FixedDelay")

    if remove_dos:
        mask &= (df["is_cheating"] == False) | (df["cheat_type"] != "DOS")

    df = df[mask].copy()
    df["iat_ms"] = df["iat"] * 1000

    # default every row in every session dataframe to False
    for key, orig_df in packet_dict.items():
        if orig_df is None or orig_df.empty:
            continue
        orig_df[detector_col] = False

    # flag rows whose IAT exceeds the fixed threshold
    flagged = df[df["iat_ms"] > gap_threshold_ms]

    for _, row in flagged.iterrows():
        key = row["_src_key"]
        idx = row["_src_idx"]
        packet_dict[key].loc[idx, detector_col] = True

    return packet_dict

In [ ]:
def extract_suspicious_window_gap_and_burst_fixed_threshold(
    packet_dict,
    gap_threshold_ms=400,
    burst_threshold_ms=1.0,
    min_fast_packets=1,
    n_following=10,
    remove_fd=False,
    remove_dos=False,
    detector_col="detector_suspicious_gap_burst_fixed",
):
    """
    Baseline lag-switch detector: "fixed-threshold (gap + burst)".

    Flags an event as positive if a gap (iat_ms > gap_threshold_ms) is
    observed AND is followed by MORE THAN `min_fast_packets` packets (within
    the next `n_following` packets in the same flow) whose IAT falls below
    a fixed burst threshold (iat_ms < burst_threshold_ms). Unlike the
    adaptive detector, both thresholds are fixed constants — no
    dk_mean/dk_stdev statistics are used.

    This mirrors the structure of extract_suspicious_window_packet_per_packet
    (gap trigger -> look at following packets -> count "fast" packets) but
    with fixed thresholds and a simple packet-count rule instead of a
    fill-ratio computed against an adaptive expected packet count.

    Adds a boolean column (`detector_col`) to every dataframe in packet_dict:
    True for every packet inside a window where the gap+burst condition was
    satisfied (the triggering gap packet plus all packets up to
    window_end), False otherwise.

    Parameters
    ----------
    packet_dict : dict[str, pd.DataFrame]
        Output of extract_suspicious_window (or any pipeline producing
        "phase", "iat", "region", "flow_id" columns).
    gap_threshold_ms : float
        Fixed IAT threshold (ms) defining a gap / event trigger.
    burst_threshold_ms : float
        Fixed IAT threshold (ms) below which a following packet counts as
        "fast" (i.e. part of the burst).
    min_fast_packets : int
        Minimum number of fast packets required within the window (strictly
        more than this count) for the event to be flagged positive. Default
        1, matching "more than one packet below the burst threshold".
    n_following : int
        Number of packets after the gap packet to inspect for fast packets
        (the gap packet itself is included in the window but is not, by
        definition, counted as a "fast" packet since it triggered the gap).
    remove_fd, remove_dos : bool
        Same semantics as extract_suspicious_window_fixed_threshold.
    detector_col : str
        Name of the boolean output column to write back into packet_dict.

    Returns
    -------
    packet_dict, mutated in place with `detector_col` added to every dataframe.
    """

    all_df = []
    source_keys = []
    source_orig_idx = []

    for key, df in packet_dict.items():
        if df is not None and not df.empty:
            all_df.append(df)
            source_keys.extend([key] * len(df))
            source_orig_idx.extend(df.index.tolist())

    if not all_df:
        print("No data available")
        return packet_dict

    df = pd.concat(all_df, ignore_index=True)
    df["_src_key"] = source_keys
    df["_src_idx"] = source_orig_idx

    mask = (
        (df["phase"] == "playing") &
        (df["iat"].notna()) &
        (df["region"].notna())
    )

    if remove_fd:
        mask &= (df["is_cheating"] == False) | (df["cheat_type"] != "FixedDelay")

    if remove_dos:
        mask &= (df["is_cheating"] == False) | (df["cheat_type"] != "DOS")

    df = df[mask].copy()
    df["iat_ms"] = df["iat"] * 1000

    # default every row in every session dataframe to False
    for key, orig_df in packet_dict.items():
        if orig_df is None or orig_df.empty:
            continue
        orig_df[detector_col] = False

    for flow_id, flow_df in df.groupby("flow_id"):
        flow_df = flow_df.reset_index(drop=True)
        iats = flow_df["iat_ms"].values
        n_rows = len(iats)

        gap_positions = np.nonzero(iats > gap_threshold_ms)[0]

        for pos in gap_positions:
            window_end = min(pos + n_following + 1, n_rows)  # +1 to include the gap packet
            following_iats = iats[pos + 1:window_end]  # exclude the gap packet itself

            n_fast = int((following_iats < burst_threshold_ms).sum())

            if n_fast > min_fast_packets:
                window_rows = flow_df.iloc[pos:window_end]
                for _, row in window_rows.iterrows():
                    key = row["_src_key"]
                    idx = row["_src_idx"]
                    packet_dict[key].loc[idx, detector_col] = True

    return packet_dict

In [ ]:
def evaluate_detector_gap_events_random_baseline(
    packet_dict,
    gap_threshold_ms=400,
    n_following=10,
    label_cheat_types=("LagSwitch",),
    timestamp_col="timestamp",
    flag_probability=0.5,
    n_trials=100,
    random_seed=None,
):
    """
    "Random baseline": for every gap event (same event construction as
    evaluate_detector_gap_events, i.e. iat_ms > gap_threshold_ms triggers an
    event spanning the gap packet plus the next n_following packets), flag
    the event as positive independently at random with probability
    `flag_probability`, rather than using any detector signal.

    Because a single random draw is noisy, this runs `n_trials` independent
    random assignments and reports the AVERAGE confusion-matrix metrics
    across trials (plus their standard deviation), which is the standard way
    to report a random baseline's expected performance.

    Parameters
    ----------
    packet_dict : dict[str, pd.DataFrame]
        Same input as evaluate_detector_gap_events. Note: a detector_col is
        NOT required for this baseline, since flags are random rather than
        read from a column.
    gap_threshold_ms, n_following, label_cheat_types, timestamp_col :
        Same semantics as evaluate_detector_gap_events — defines how events
        are constructed and labeled, so the random baseline is evaluated on
        the exact same set of events as the real detector.
    flag_probability : float
        Probability of flagging each event as positive (default 0.5, i.e. a
        coin flip; set to the empirical positive rate among events for a
        "stratified random" baseline if preferred).
    n_trials : int
        Number of independent random trials to average over.
    random_seed : int or None
        Seed for reproducibility. Each trial uses a different sub-seed
        derived from this base seed.

    Returns
    -------
    dict with keys:
        "mean" : dict of mean tp/fp/tn/fn/precision/recall/f1/accuracy
                 across trials
        "std"  : dict of standard deviation of the same metrics across trials
        "trials" : pd.DataFrame, one row per trial, with all metrics
        "events" : the underlying event table (ground_truth column only;
                   no single "flagged" column since flags are randomized per trial)
    """
    valid_items = [(key, df) for key, df in packet_dict.items() if df is not None and not df.empty]

    if not valid_items:
        print("No data available")
        return None

    sample_df = valid_items[0][1]
    for required_col in ("flow_id", "iat", timestamp_col):
        if required_col not in sample_df.columns:
            raise ValueError(f"Column '{required_col}' not found in packet dataframes")

    # --- build the per-event ground truth table (no detector flags needed) ---
    event_rows = []
    for session_key, session_df in valid_items:
        gt_mask = _ground_truth(session_df, label_cheat_types)

        for flow_id, flow_df in session_df.groupby("flow_id"):
            flow_df = flow_df.sort_values(timestamp_col)
            flow_idx = flow_df.index.to_numpy()
            n_rows = len(flow_idx)

            iat_ms = flow_df["iat"].values * 1000
            gap_positions = np.nonzero(iat_ms > gap_threshold_ms)[0]

            if len(gap_positions) == 0:
                continue

            flow_gt = gt_mask.loc[flow_idx].values

            for pos in gap_positions:
                window_end = min(pos + n_following + 1, n_rows)
                window_labeled = bool(flow_gt[pos:window_end].any())
                event_rows.append({
                    "session": session_key,
                    "flow_id": flow_id,
                    "gap_position": pos,
                    "ground_truth": window_labeled,
                })

    if not event_rows:
        print("No gap events found above threshold")
        return None

    events_df = pd.DataFrame(event_rows)
    n_events = len(events_df)
    gt = events_df["ground_truth"].values

    rng = np.random.default_rng(random_seed)

    trial_rows = []
    for trial in range(n_trials):
        predicted = rng.random(n_events) < flag_probability
        m = _confusion_metrics(pd.Series(predicted), pd.Series(gt))
        trial_rows.append(m)

    trials_df = pd.DataFrame(trial_rows)

    mean_metrics = trials_df.mean().to_dict()
    std_metrics = trials_df.std().to_dict()

    return {
        "mean": mean_metrics,
        "std": std_metrics,
        "trials": trials_df,
        "events": events_df,
    }

### 2.10 Adaptive detector with a half-life chosen per ISP

In [ ]:
def run_adaptive_half_life_detector(
    pcap_dict,
    best_half_life_per_isp,
    isp_col="isp",
    default_half_life=None,
    gap_threshold_min_ms=400,
    burst_threshold_max_ms=1.0,
    suspicious_fill_ratio_threshold=0.5,
    gap_threshold_ms=400,
    n_following=10,
    label_cheat_types=("LagSwitch",),
    per_region=True,
    per_isp=True,
):
    """
    Run the detector using a DIFFERENT half_life per ISP, rather than one
    global half_life for everyone, then evaluate the combined result at the
    event level (evaluate_detector_gap_events).

    Since a single session in pcap_dict can contain packets from more than
    one ISP, each session's dataframe is first split into per-ISP row
    chunks, decay_iat_features is applied to each chunk with that ISP's own
    half_life, and the chunks are recombined into a single dataframe per
    session before running the detector. This assumes flow_id does not span
    multiple ISPs (i.e. a flow's packets all belong to the same ISP) — if a
    flow's IAT-derived features were computed half "as ISP A" and half "as
    ISP B", the adaptive smoothing would be inconsistent within that flow's
    own decay state, so this is flagged via a sanity check below rather than
    silently assumed.

    Parameters
    ----------
    pcap_dict : dict[str, pd.DataFrame]
        Raw packet_dict, BEFORE feature extraction (i.e. before
        decay_iat_features has been applied). Each dataframe must have an
        `isp_col` column.
    best_half_life_per_isp : dict[isp -> half_life] or pd.DataFrame
        Either a plain dict mapping each ISP to its chosen half_life (e.g.
        built from find_best_half_life_per_isp's output), or that function's
        output DataFrame directly (must have `isp_col` and "best_half_life"
        columns) — both forms are accepted.
    isp_col : str
        Column identifying each packet's ISP.
    default_half_life : float or None
        half_life to use for any ISP not present in best_half_life_per_isp
        (e.g. an ISP with too little data to have been swept). If None,
        rows belonging to an unmapped ISP are dropped with a printed warning.
    gap_threshold_min_ms, burst_threshold_max_ms, suspicious_fill_ratio_threshold :
        Passed through to extract_suspicious_window_packet_per_packet.
    gap_threshold_ms, n_following, label_cheat_types :
        Passed through to evaluate_detector_gap_events.
    per_region, per_isp : bool
        Passed through to evaluate_detector_gap_events.

    Returns
    -------
    dict with keys:
        "half_life_used_per_isp" : pd.DataFrame, one row per ISP showing the
            half_life actually applied (i.e. the resolved mapping, including
            any default_half_life fallbacks used)
        "results" : the full dict returned by evaluate_detector_gap_events
            on the combined adaptive-half-life detector output (includes
            "total", "per_region", "per_isp", "events")
        "detector_output_pcap" : the processed packet_dict (with decay
            features computed per-ISP and detector_suspicious annotated),
            in case you want to inspect or re-evaluate it differently
    """
    # normalize best_half_life_per_isp into a plain dict
    if isinstance(best_half_life_per_isp, pd.DataFrame):
        rate_map = dict(zip(
            best_half_life_per_isp[isp_col],
            best_half_life_per_isp["best_half_life"],
        ))
    else:
        rate_map = dict(best_half_life_per_isp)

    # --- resolve the half_life actually used per ISP (including fallbacks) ---
    all_isps_seen = set()
    for df in pcap_dict.values():
        if df is not None and not df.empty and isp_col in df.columns:
            all_isps_seen.update(df[isp_col].dropna().unique().tolist())

    resolved_half_life = {}
    dropped_isps = []
    for isp in all_isps_seen:
        if isp in rate_map:
            resolved_half_life[isp] = rate_map[isp]
        elif default_half_life is not None:
            resolved_half_life[isp] = default_half_life
        else:
            dropped_isps.append(isp)

    if dropped_isps:
        print(
            f"Dropping {len(dropped_isps)} ISP(s) with no half_life mapping and no "
            f"default_half_life provided: {dropped_isps}"
        )

    half_life_used_df = pd.DataFrame([
        {isp_col: isp, "half_life_used": hl} for isp, hl in resolved_half_life.items()
    ]).sort_values(isp_col).reset_index(drop=True)

    # --- split each session by ISP, apply that ISP's half_life, recombine ---
    processed = {}
    for session_key, session_df in pcap_dict.items():
        if session_df is None or session_df.empty:
            processed[session_key] = session_df
            continue

        if isp_col not in session_df.columns:
            raise ValueError(f"Column '{isp_col}' not found in session {session_key!r}")

        # sanity check: a flow should not span multiple ISPs, otherwise
        # per-ISP half_life smoothing is inconsistent within that flow
        if "flow_id" in session_df.columns:
            isp_per_flow = session_df.groupby("flow_id")[isp_col].nunique()
            bad_flows = isp_per_flow[isp_per_flow > 1]
            if len(bad_flows) > 0:
                print(
                    f"WARNING: session {session_key!r} has {len(bad_flows)} flow(s) "
                    f"spanning multiple ISPs (flow_ids: {bad_flows.index.tolist()}); "
                    "decay features for these flows will be inconsistent across the "
                    "ISP boundary."
                )

        isp_chunks = []
        for isp, isp_chunk in session_df.groupby(isp_col, dropna=False):
            if isp not in resolved_half_life:
                continue  # dropped, no mapping and no default
            hl = resolved_half_life[isp]
            isp_chunks.append(decay_iat_features(isp_chunk.copy(), half_life=hl))

        if not isp_chunks:
            processed[session_key] = session_df.iloc[0:0]  # empty, all ISPs dropped
            continue

        processed[session_key] = pd.concat(isp_chunks, ignore_index=False).sort_index()

    # --- run the detector on the recombined, per-ISP-smoothed packet_dict ---
    detector_output_pcap = extract_suspicious_window_packet_per_packet(
        processed,
        gap_threshold_min_ms=gap_threshold_min_ms,
        burst_threshold_max_ms=burst_threshold_max_ms,
        suspicious_fill_ratio_threshold=suspicious_fill_ratio_threshold,
    )

    # --- evaluate at the event level ---
    results = evaluate_detector_gap_events(
        detector_output_pcap,
        gap_threshold_ms=gap_threshold_ms,
        n_following=n_following,
        label_cheat_types=label_cheat_types,
        per_region=per_region,
        per_isp=per_isp,
        isp_col=isp_col,
        per_flow=True
    )

    return {
        "half_life_used_per_isp": half_life_used_df,
        "results": results,
        "detector_output_pcap": detector_output_pcap,
    }

### 2.11 Comparing detectors via likelihood ratios

In [ ]:
def compute_fpr_fnr_likelihood_ratios(metrics):
    """
    Given a confusion-matrix metrics dict (must have "tp", "fp", "tn", "fn"
    keys — e.g. results["total"] from evaluate_detector_gap_events, or any
    other evaluate_* function), compute:
 
        TPR  = tp / (tp + fn)                  True Positive Rate (recall)
        FPR  = fp / (fp + tn)                  False Positive Rate
        FNR  = fn / (tp + fn)                  False Negative Rate
        LR+  = TPR / FPR                        Positive likelihood ratio
        LR-  = FNR / TNR  = FNR / (1 - FPR)     Negative likelihood ratio
 
    Precision and F1 are passed through unchanged from the input `metrics`
    if present (e.g. as already computed by _confusion_metrics), rather than
    recomputed here.
 
    LR+ and LR- summarize how much a positive (resp. negative) detector
    verdict shifts the odds of actual cheating, independent of the base
    rate. LR+ >> 1 means a positive flag is strong evidence of cheating;
    LR- close to 0 means a negative flag is strong evidence of innocence.
 
    Parameters
    ----------
    metrics : dict
        Either a flat dict with "tp"/"fp"/"tn"/"fn" keys (optionally also
        "precision"/"f1"), or a results dict with a "total" key holding such
        a dict (accepted for convenience so you can pass
        evaluate_detector_gap_events(...)'s return value directly without
        first drilling into ["total"]).
 
    Returns
    -------
    dict with keys: tpr, fpr, fnr, tnr, precision, f1, lr_plus, lr_minus.
    lr_plus is None if fpr == 0 (undefined, division by zero); lr_minus is
    None if tnr == 0 for the same reason. precision/f1 are None if not
    present in the input metrics.
    """
    if "total" in metrics and "tp" not in metrics:
        metrics = metrics["total"]
 
    tp = metrics["tp"]
    fp = metrics["fp"]
    tn = metrics["tn"]
    fn = metrics["fn"]
 
    fpr = fp / (fp + tn) if (fp + tn) > 0 else 0.0
    fnr = fn / (tp + fn) if (tp + fn) > 0 else 0.0
    tpr = 1 - fnr  # == recall
    tnr = 1 - fpr
 
    lr_plus = (tpr / fpr) if fpr > 0 else None
    lr_minus = (fnr / tnr) if tnr > 0 else None
 
    return {
        "tpr": tpr,
        "fpr": fpr,
        "fnr": fnr,
        "tnr": tnr,
        "precision": metrics.get("precision"),
        "f1": metrics.get("f1"),
        "lr_plus": lr_plus,
        "lr_minus": lr_minus,
    }

In [ ]:
def compare_baselines_fpr_fnr_lr(results_by_baseline):
    """
    Apply compute_fpr_fnr_likelihood_ratios to several baselines/detectors at
    once and return a single comparison table, formatted to match:
 
        | Detector | TPR | FPR | Precision | F1 | LR+ | LR- |
 
    Parameters
    ----------
    results_by_baseline : dict[str -> dict]
        Maps a baseline name (e.g. "Fixed threshold (gap)", "Random",
        "Fixed threshold (gap + burst)", "Adaptive") to that baseline's
        results dict (or just its "total" sub-dict — both accepted, same as
        compute_fpr_fnr_likelihood_ratios).
 
    Returns
    -------
    pd.DataFrame, one row per baseline, columns: Detector, TPR, FPR,
    Precision, F1, LR+, LR- (in that order).
    """
    rows = []
    for name, metrics in results_by_baseline.items():
        m = compute_fpr_fnr_likelihood_ratios(metrics)
        rows.append({
            "Detector": name,
            "TPR": m["tpr"],
            "FPR": m["fpr"],
            "Precision": m["precision"],
            "F1": m["f1"],
            "LR+": m["lr_plus"],
            "LR-": m["lr_minus"],
        })
 
    return pd.DataFrame(rows)

### Run: sweep half-lives per ISP, then run the adaptive detector

In [ ]:
# 1. Sweep half-lives per ISP
half_life_values = np.logspace(np.log10(1), np.log10(40), num=10)
sweep_df, _ = compare_half_life_impact_detection_gap_events_per_isp(
    pcap_inthewild, half_life_values=half_life_values,
    label_cheat_types=("LagSwitch",),
)
best_per_isp = find_best_half_life_per_isp(sweep_df, metric="f1")

# 2. Run the detector with each ISP using its own best half-life, and evaluate
adaptive_run = run_adaptive_half_life_detector(
    pcap_inthewild,
    best_half_life_per_isp=best_per_isp,   # accepts the DataFrame directly
    isp_col="isp",
    default_half_life=20,  # fallback for any ISP the sweep didn't cover
    gap_threshold_ms=400,
    n_following=3,
    label_cheat_types=("LagSwitch",),
)

print(adaptive_run["half_life_used_per_isp"])    # which half_life was applied per ISP
print(adaptive_run["results"]["total"])          # aggregated event-level TP/FP/TN/FN/F1/etc.
print(adaptive_run["results"]["per_isp"])        # per-ISP breakdown under the adaptive scheme

### Baselines

In [ ]:
# Fixed-threshold on gap 
gap_baseline_output = extract_suspicious_window_fixed_threshold(pcap_inthewild, gap_threshold_ms=400)
gap_baseline_results = evaluate_detector_gap_events(gap_baseline_output, detector_col="detector_suspicious_fixed")

# Fixed-threshold on gap + burst
gap_burst_output = extract_suspicious_window_gap_and_burst_fixed_threshold(pcap_inthewild, gap_threshold_ms=400, burst_threshold_ms=1.0, min_fast_packets=1)
gap_burst_results = evaluate_detector_gap_events(gap_burst_output, detector_col="detector_suspicious_gap_burst_fixed")

# Random baseline
random_results = evaluate_detector_gap_events_random_baseline(pcap_inthewild, gap_threshold_ms=400, n_following=10, n_trials=100, random_seed=42)

Results per-event, adaptive detector vs. all baselines:

In [ ]:
comparison = compare_baselines_fpr_fnr_lr({
    "Random": random_results["mean"],
    "Fixed threshold (gap)": gap_baseline_results,
    "Fixed threshold (gap + burst)": gap_burst_results,
    "Adaptive": adaptive_run["results"]["total"],
})

display(comparison)

### Evaluate per player

In [ ]:
# extract results per flow_id
results_per_flow = adaptive_run["results"]["per_flow"]
results_per_flow["Cheat"] = (results_per_flow["tp"] ) | (results_per_flow["fn"] )

A simple (non-mixture-model) log-likelihood-ratio score per flow, swept over a range of thresholds `tau`:

In [ ]:
def compute_log_lr(row, lr_plus, lr_minus):
    """Event-level log-likelihood-ratio for one flow, given its TP/FN counts."""
    tp = row["tp"]
    fn = row["fn"]
    return tp * np.log(lr_plus) + fn * np.log(lr_minus)


def evaluate_lr_thresholds(df, lr_col, y_true_col, taus):
    """Sweep decision thresholds `tau` on `df[lr_col]` and score against `df[y_true_col]`."""
    results = []
    for tau in taus:
        y_pred = (df[lr_col] > tau).astype(int)
        y_true = df[y_true_col]

        tp = int(((y_pred == 1) & (y_true == 1)).sum())
        fp = int(((y_pred == 1) & (y_true == 0)).sum())
        tn = int(((y_pred == 0) & (y_true == 0)).sum())
        fn = int(((y_pred == 0) & (y_true == 1)).sum())

        precision = tp / (tp + fp) if (tp + fp) > 0 else 0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0
        f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) > 0 else 0
        acc = (tp + tn) / (tp + tn + fp + fn) if (tp + tn + fp + fn) > 0 else 0

        results.append({
            "tau": tau, "TP": tp, "FP": fp, "TN": tn, "FN": fn,
            "precision": precision, "recall": recall, "f1": f1, "accuracy": acc,
        })
    return pd.DataFrame(results)

In [ ]:
adaptive_run["results"]["total"]

In [ ]:
# Event-level likelihood ratios from the adaptive detector
m = compute_fpr_fnr_likelihood_ratios(adaptive_run["results"]["total"])
LR_plus = m["lr_plus"]  # TP-driven evidence
LR_minus = m["lr_minus"]  # FN/missed-event penalty

results_per_flow["log_lr"] = results_per_flow.apply(compute_log_lr, axis=1, args=(LR_plus, LR_minus))
results_per_flow["lr"] = np.exp(results_per_flow["log_lr"])
results_per_flow["y_true"] = (results_per_flow["Cheat"] > 0).astype(int)

taus = np.logspace(np.log10(1), np.log10(100), num=10)
results_results_per_flow = evaluate_lr_thresholds(results_per_flow, lr_col="lr", y_true_col="y_true", taus=taus)

print(results_results_per_flow)

Aggregate by player (base name), across all of a player's IPs/flows:

In [ ]:
pcap_players = extract_players_multiple_sessions(PATH_LOGS)
pcap_players = pcap_players[0]
df_players_all = pd.concat(pcap_players.values(), ignore_index=True)

In [ ]:
ip_map = (
    df_players_all.groupby("base_name")["ip_port"]
      .agg(list)
      .to_dict()
)

Map each flow's `ip_port` back to a player base name:

In [ ]:
def extract_other_endpoint(flow_id):
    left, right = flow_id.split("-")
    
    if left.startswith(SERVER_IP):
        return right
    else:
        return left

In [ ]:
results_per_player = adaptive_run["results"]["per_flow"].copy()
results_per_player = results_per_player[['flow_id', 'tp', 'fp', 'tn', 'fn', 'total', 'precision', 'recall', 'f1', 'accuracy', 'Cheat']]
results_per_player["ip_port"] = results_per_player["flow_id"].apply(extract_other_endpoint)

In [ ]:
ip_map_clean = {
    k: [x for x in v if x and ":" in x]
    for k, v in ip_map.items()
}

ip_map_clean = {
    k: sorted(set(v))
    for k, v in ip_map_clean.items()
}

ip_to_base = {
    ip: base
    for base, ips in ip_map_clean.items()
    for ip in ips
}

In [ ]:
results_per_player["base_name"] = results_per_player["ip_port"].map(ip_to_base)


results_per_player = results_per_player.dropna(subset=["base_name"])

agg_base = (
    results_per_player
    .groupby("base_name")
    .agg(
        ip_ports=("ip_port", lambda x: sorted(set(x))),
        n_flows=("flow_id", "count"),

        tp=("tp", "sum"),
        fp=("fp", "sum"),
        tn=("tn", "sum"),
        fn=("fn", "sum"),
        Cheat=("Cheat", "sum"),
        total=("total", "sum"),

        cheat_rate=("Cheat", "mean"),
    )
    .reset_index()
)

In [ ]:
m = compute_fpr_fnr_likelihood_ratios(adaptive_run["results"]["total"])
LR_plus = m["lr_plus"]  # TP-driven evidence
LR_minus = m["lr_minus"]  # FN/missed-event penalty

agg_base["log_lr"] = agg_base.apply(compute_log_lr, axis=1, args=(LR_plus, LR_minus))
agg_base["lr"] = np.exp(agg_base["log_lr"])
agg_base["y_true"] = (agg_base["Cheat"] > 0).astype(int)

taus = [1, 3, 10]
results_agg_base = evaluate_lr_thresholds(agg_base, lr_col="lr", y_true_col="y_true", taus=taus)

display(results_agg_base)